In [1]:
import os
os.environ["HF_HOME"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf"
os.environ["HF_HUB_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers"
os.environ["HF_HUB_OFFLINE"] = "1"   # hard offline, raises if not found locally

from transformers import AutoProcessor, SiglipVisionModel

model_id = "google/siglip-so400m-patch14-384"
processor = AutoProcessor.from_pretrained(model_id, local_files_only=True)
vision    = SiglipVisionModel.from_pretrained(model_id, local_files_only=True)


/media/pc1/Ubuntu/Extend_Data/ngoc/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
import os, json, math, random
from pathlib import Path
import numpy as np
import torch, pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

# ---------- Config
ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
DATA_CSV = ROOT / "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated_safe.csv"
SCHEMA_JSON = DATA_CSV.parent / "labels_schema_filtered_safe.json"

TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
SIGLIP_PATH = str(next((p for p in SIGLIP_DIR.iterdir() if p.is_dir()), TRANSF_CACHE / "models--google--siglip-so400m-patch14-384"))

# Offline caches
os.environ.setdefault("HF_HOME", str(ROOT / "hf"))
os.environ.setdefault("HF_HUB_CACHE", str(ROOT / "hf" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf" / "transformers"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# Device/dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
USE_AMP = (device == "cuda")

print("Loading model from:", SIGLIP_PATH)
processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# ---------- Data
labels = json.loads(Path(SCHEMA_JSON).read_text())
df = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False, low_memory=False)

# Only keep rows with existing file to avoid skew
def path_exists(p): 
    try: 
        return Path(p).exists()
    except: 
        return False

if "path" not in df.columns:
    raise ValueError("CSV must have a 'path' column.")
df = df[df["path"].map(path_exists)]
print(f"[INFO] Rows with existing files: {len(df)}")

# ---------- Prompts (baseline: 'detailed')
DETAILED_PROMPTS = {
    "qr": "a square black and white QR code",
    "logo": "a company or journal logo",
    "journal_banner": "a scientific journal article header",
    "pure_text": "a page with only text paragraphs",
    "book_cover": "a book cover with title text",
    "doc_fullpage": "a complete formatted document page",
    "empty": "an almost empty white page",
    "ecg": "an electrocardiogram waveform",
    "profile": "a portrait photo of a person",
    "gel_electrophoresis": "a gel with DNA or protein bands",
    "map": "a geographic map",
    "tiny": "a very small thumbnail image",
    "house": "a house or building",
    "illustration": "a diagram or illustration",
    "phylogenetic_tree": "a phylogenetic tree",
    "chart": "a data chart or graph",
}

# Optional additional variants for MAX-pooling (short + detailed). Keep small to avoid flattening.
VARIANTS = {
    "qr": ["a QR code"],
    "logo": ["a logo"],
    "journal_banner": ["journal header with title and authors"],
    "pure_text": ["text paragraphs only"],
    "doc_fullpage": ["full document page"],
    "gel_electrophoresis": ["gel electrophoresis bands"],
    "chart": ["data chart graph"],
    # add more if desired
}

def prompts_for_label(lbl):
    p = [ DETAILED_PROMPTS.get(lbl, f"a {lbl.replace('_',' ')}") ]
    p += VARIANTS.get(lbl, [])
    return p

# ---------- Text embeddings with MAX pooling at score time
# We keep all prompt embeddings; during scoring we take max(sim) over a label's variants.
with torch.no_grad():
    all_prompts = []
    label_ranges = {}
    idx = 0
    for lbl in labels:
        ps = prompts_for_label(lbl)
        label_ranges[lbl] = (idx, idx+len(ps))
        all_prompts.extend(ps)
        idx += len(ps)
    print(f"[INFO] Total prompts: {len(all_prompts)}")

    text_inputs = processor(text=all_prompts, return_tensors="pt", padding=True, truncation=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            text_emb_all = model.get_text_features(**text_inputs)
    else:
        text_emb_all = model.get_text_features(**text_inputs)
    text_emb_all = torch.nn.functional.normalize(text_emb_all, dim=-1).to(DTYPE)
    del text_inputs

# Build an index so we can gather a matrix for each label quickly
label_prompt_mats = {}  # lbl -> [n_prompts_lbl, d]
for lbl in labels:
    a, b = label_ranges[lbl]
    label_prompt_mats[lbl] = text_emb_all[a:b]

# ---------- Image embedder with light TTA
MIN_SIDE = 16
def load_rgb(path):
    try:
        im = Image.open(path).convert("RGB")
        w,h = im.size
        if min(w,h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w,h))
            im = im.resize((max(MIN_SIDE,int(w*scale)), max(MIN_SIDE,int(h*scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

def embed_images(paths, tta="center+shrink"):  # 'none' | 'center+shrink' | 'three_sizes'
    ims = [load_rgb(p) for p in paths]
    valid = [im is not None for im in ims]
    ims_valid = [im for im in ims if im is not None]
    if not ims_valid:
        return None, valid

    with torch.no_grad():
        embs = []
        def _proc(imgs):
            inputs = processor(images=imgs, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    e = model.get_image_features(**inputs)
            else:
                e = model.get_image_features(**inputs)
            e = torch.nn.functional.normalize(e, dim=-1).to(DTYPE)
            return e

        if tta == "none":
            embs.append(_proc(ims_valid))
        elif tta == "center+shrink":
            # 2 views: normal; slight downscale to induce a different resize/crop path
            embs.append(_proc(ims_valid))
            ims2 = [im.resize((int(im.width*0.9), int(im.height*0.9)), Image.BILINEAR) for im in ims_valid]
            embs.append(_proc(ims2))
        elif tta == "three_sizes":
            for s in (1.0, 0.9, 1.1):
                ims_s = [im.resize((int(im.width*s), int(im.height*s)), Image.BILINEAR) for im in ims_valid]
                embs.append(_proc(ims_s))
        else:
            embs.append(_proc(ims_valid))

        img_emb = torch.stack(embs, dim=0).mean(0)  # average TTA
    return img_emb, valid

# ---------- MAX-pool scoring across each label's prompts
def text_logits_max(img_emb):
    # returns [n_img, n_labels] logits
    outs = []
    for lbl in labels:
        T = label_prompt_mats[lbl]              # [m, d]
        sims = img_emb @ T.T                    # [n, m]
        mvals, _ = sims.max(dim=1)              # [n]
        outs.append(mvals.unsqueeze(1))
    return torch.cat(outs, dim=1)               # [n, C]

# ---------- Tip-Adapter-like prototype boost (no training)
def build_prototypes(K=8, per_class_min=1, seed=0):
    rng = np.random.RandomState(seed)
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    if not present:
        raise ValueError("No label_* columns in CSV.")

    # pick shots per class (only rows that have the label==1)
    prot_paths = []
    prot_targets = []  # class index
    for ci, l in enumerate(labels):
        col = f"label_{l}"
        if col not in df.columns:
            continue
        pos_rows = df[df[col].astype(str).fillna("0").isin(["1","1.0"])]
        pos_rows = pos_rows[pos_rows["path"].map(path_exists)]
        if len(pos_rows) < per_class_min:
            continue
        take = min(K, len(pos_rows))
        chosen = pos_rows.sample(take, random_state=seed)["path"].tolist()
        prot_paths.extend(chosen)
        prot_targets.extend([ci]*take)

    if not prot_paths:
        print("[WARN] No prototypes built.")
        return None, None

    # embed prototype images (with same TTA)
    P_emb, valid = embed_images(prot_paths, tta="center+shrink")
    valid_idx = [i for i,v in enumerate(valid) if v]
    P_emb = P_emb[valid_idx]
    prot_targets = [prot_targets[i] for i in valid_idx]

    # build class-onehot for KNN logits
    C = len(labels)
    oh = torch.zeros((len(prot_targets), C), dtype=DTYPE, device=device)
    for i, ci in enumerate(prot_targets):
        oh[i, ci] = 1.0
    return P_emb, oh

def prototype_logits(img_emb, P_emb, oh, beta=80.0):
    # cosine distance -> exp kernel
    sims = img_emb @ P_emb.T                              # [n, np]
    # convert to distance (1 - cos) then RBF-like: exp(beta * sim) also works.
    # We'll just sharpen sims directly:
    knn = torch.softmax(beta * sims, dim=1)              # [n, np]
    logits = knn @ oh                                    # [n, C]
    return logits

# ---------- Master scorer
class Scorer:
    def __init__(self, use_max_text=True, use_prototypes=True, alpha=0.7, beta=80.0, tta_mode="center+shrink", shots=8):
        self.use_max_text = use_max_text
        self.use_prototypes = use_prototypes
        self.alpha = alpha
        self.beta = beta
        self.tta_mode = tta_mode
        self.P = None
        self.OH = None
        if use_prototypes:
            self.P, self.OH = build_prototypes(K=shots, per_class_min=1, seed=0)
            if self.P is None:
                self.use_prototypes = False
                print("[INFO] Falling back to text-only (no prototypes).")

    def score_paths(self, paths):
        img_emb, valid = embed_images(paths, tta=self.tta_mode)
        if img_emb is None:
            return None, valid

        # text logits
        if self.use_max_text:
            text_log = text_logits_max(img_emb)
        else:
            # fallback: single detailed prompt per label
            # (not used; you can wire it if you wish)
            text_log = text_logits_max(img_emb)

        # prototype logits
        if self.use_prototypes and self.P is not None:
            proto_log = prototype_logits(img_emb, self.P, self.OH, beta=self.beta)
            final = self.alpha * text_log + (1.0 - self.alpha) * proto_log
        else:
            final = text_log

        return final, valid

# ---------- Evaluation (image-level & per-class)
def evaluate(scorer, sample_size=64, seed=0, batch=16):
    rng = np.random.RandomState(seed)
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    assert present, "No label_* columns."

    # sample rows with at least one positive
    tmp = df[df[present].apply(lambda r: any(x in ["1","1.0",1,1.0,True] for x in r), axis=1)]
    if len(tmp) == 0:
        raise ValueError("No positive rows to evaluate.")
    n = min(sample_size, len(tmp))
    sample = tmp.sample(n, random_state=seed).reset_index(drop=True)

    paths = sample["path"].tolist()
    img_cnt = img_hit1 = img_hit5 = 0

    hit1 = {l: 0 for l in labels}
    hit5 = {l: 0 for l in labels}
    cnt  = {l: 0 for l in labels}

    for i in tqdm(range(0, len(paths), batch), desc="Scoring"):
        chunk = paths[i:i+batch]
        logits, valid = scorer.score_paths(chunk)
        if logits is None:
            continue
        logits_cpu = logits.detach().cpu()
        row_slice = sample.iloc[i:i+batch]

        k = 0
        for j, (_, r) in enumerate(row_slice.iterrows()):
            if not valid[j]:
                continue
            row_log = logits_cpu[k]; k += 1
            top = torch.argsort(row_log, descending=True).tolist()
            top5 = [labels[idx] for idx in top[:5]]
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,1.0,True]]

            # image-level
            img_cnt += 1
            img_hit1 += int(labels[top[0]] in gt)
            img_hit5 += int(any(t in gt for t in top5))

            # per-class (as you had)
            for g in gt:
                cnt[g] += 1
                if labels[top[0]] == g:
                    hit1[g] += 1
                if g in top5:
                    hit5[g] += 1

    # Summaries
    print("\n=== IMAGE-LEVEL METRICS (multi-label aware) ===")
    print(f"Hit@1: {img_hit1}/{img_cnt} ({img_hit1/img_cnt:.1%})")
    print(f"Hit@5: {img_hit5}/{img_cnt} ({img_hit5/img_cnt:.1%})")

    print("\n=== PER-CLASS (your original style) ===")
    for l in labels:
        if cnt[l] > 0:
            print(f"{l:22s}: H@1={hit1[l]}/{cnt[l]}  H@5={hit5[l]}/{cnt[l]}")

# ---------- Run
scorer = Scorer(
    use_max_text=True,       # MAX-pool across a few strong prompts
    use_prototypes=True,     # Tip-Adapter-like KNN boost
    alpha=0.7,               # mix weight: 0.7 text / 0.3 prototypes
    beta=80.0,               # kernel sharpness for prototypes
    tta_mode="center+shrink",# 2-view TTA
    shots=8                  # up to 8 prototypes per class
)

evaluate(scorer, sample_size=64, seed=0, batch=16)

Loading model from: /media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers/models--google--siglip-so400m-patch14-384/snapshots/9fdffc58afc957d1a03a25b10dba0329ab15c2a3
[INFO] Rows with existing files: 163
[INFO] Total prompts: 23


Scoring: 100%|██████████| 4/4 [00:04<00:00,  1.24s/it]


=== IMAGE-LEVEL METRICS (multi-label aware) ===
Hit@1: 54/64 (84.4%)
Hit@5: 64/64 (100.0%)

=== PER-CLASS (your original style) ===
qr                    : H@1=3/3  H@5=3/3
logo                  : H@1=7/10  H@5=10/10
chart                 : H@1=2/2  H@5=2/2
journal_banner        : H@1=13/19  H@5=19/19
pure_text             : H@1=4/4  H@5=4/4
empty                 : H@1=7/7  H@5=7/7
ecg                   : H@1=1/1  H@5=1/1
profile               : H@1=1/2  H@5=2/2
gel_electrophoresis   : H@1=2/2  H@5=2/2
map                   : H@1=3/3  H@5=3/3
tiny                  : H@1=6/6  H@5=6/6
house                 : H@1=1/1  H@5=1/1
doc_fullpage          : H@1=4/4  H@5=4/4


In [6]:
# ==========================================
# v4.1: MAX-pool + TTA + Prototypes
#       + A) Export CSV
#       + B) Per-class alpha mix
#       + C) Doc-cluster reranker
#       + D) Simple grid search
# ==========================================

import os, json, math, random
from pathlib import Path
import numpy as np
import torch, pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

# ---------- Config
ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
DATA_CSV = ROOT / "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated_safe.csv"
SCHEMA_JSON = DATA_CSV.parent / "labels_schema_filtered_safe.json"

TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
try:
    SIGLIP_PATH = str(next(p for p in SIGLIP_DIR.iterdir() if p.is_dir()))
except StopIteration:
    SIGLIP_PATH = str(TRANSF_CACHE / "models--google--siglip-so400m-patch14-384")

# Offline caches
os.environ.setdefault("HF_HOME", str(ROOT / "hf"))
os.environ.setdefault("HF_HUB_CACHE", str(ROOT / "hf" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf" / "transformers"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# Device/dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
USE_AMP = (device == "cuda")

print("Loading model from:", SIGLIP_PATH)
processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# ---------- Data
labels = json.loads(Path(SCHEMA_JSON).read_text())
df = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False, low_memory=False)

# Only keep rows with existing file to avoid skew
def path_exists(p):
    try:
        return Path(p).exists()
    except Exception:
        return False

if "path" not in df.columns:
    raise ValueError("CSV must have a 'path' column.")
df = df[df["path"].map(path_exists)]
print(f"[INFO] Rows with existing files: {len(df)}")

# ---------- Prompts (baseline: 'detailed') + small variants for MAX-pool
DETAILED_PROMPTS = {
    "qr": "a square black and white QR code",
    "logo": "a company or journal logo",
    "journal_banner": "a scientific journal article header",
    "pure_text": "a page with only text paragraphs",
    "book_cover": "a book cover with title text",
    "doc_fullpage": "a complete formatted document page",
    "empty": "an almost empty white page",
    "ecg": "an electrocardiogram waveform",
    "profile": "a portrait photo of a person",
    "gel_electrophoresis": "a gel with DNA or protein bands",
    "map": "a geographic map",
    "tiny": "a very small thumbnail image",
    "house": "a house or building",
    "illustration": "a diagram or illustration",
    "phylogenetic_tree": "a phylogenetic tree",
    "chart": "a data chart or graph",
}

VARIANTS = {
    "qr": ["a QR code"],
    "logo": ["a logo"],
    "journal_banner": ["journal header with title and authors"],
    "pure_text": ["text paragraphs only"],
    "doc_fullpage": ["full document page"],
    "gel_electrophoresis": ["gel electrophoresis bands"],
    "chart": ["data chart graph"],
    # add more if desired (keep tight: 1–2 extras per label)
}

def prompts_for_label(lbl):
    p = [DETAILED_PROMPTS.get(lbl, f"a {lbl.replace('_',' ')}")]
    p += VARIANTS.get(lbl, [])
    return p

# ---------- Text embeddings with MAX pooling at score time
with torch.no_grad():
    all_prompts = []
    label_ranges = {}
    idx = 0
    for lbl in labels:
        ps = prompts_for_label(lbl)
        label_ranges[lbl] = (idx, idx + len(ps))
        all_prompts.extend(ps)
        idx += len(ps)
    print(f"[INFO] Total prompts: {len(all_prompts)}")

    text_inputs = processor(text=all_prompts, return_tensors="pt", padding=True, truncation=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            text_emb_all = model.get_text_features(**text_inputs)
    else:
        text_emb_all = model.get_text_features(**text_inputs)
    text_emb_all = torch.nn.functional.normalize(text_emb_all, dim=-1).to(DTYPE)
    del text_inputs

label_prompt_mats = {}  # lbl -> [m, d]
for lbl in labels:
    a, b = label_ranges[lbl]
    label_prompt_mats[lbl] = text_emb_all[a:b]

# ---------- Image embedder with light TTA
MIN_SIDE = 16
def load_rgb(path):
    try:
        im = Image.open(path).convert("RGB")
        w, h = im.size
        if min(w, h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w, h))
            im = im.resize((max(MIN_SIDE, int(w * scale)), max(MIN_SIDE, int(h * scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

def embed_images(paths, tta="center+shrink"):  # 'none' | 'center+shrink' | 'three_sizes'
    ims = [load_rgb(p) for p in paths]
    valid = [im is not None for im in ims]
    ims_valid = [im for im in ims if im is not None]
    if not ims_valid:
        return None, valid

    with torch.no_grad():
        embs = []

        def _proc(imgs):
            inputs = processor(images=imgs, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    e = model.get_image_features(**inputs)
            else:
                e = model.get_image_features(**inputs)
            e = torch.nn.functional.normalize(e, dim=-1).to(DTYPE)
            return e

        if tta == "none":
            embs.append(_proc(ims_valid))
        elif tta == "center+shrink":
            embs.append(_proc(ims_valid))
            ims2 = [im.resize((int(im.width * 0.9), int(im.height * 0.9)), Image.BILINEAR) for im in ims_valid]
            embs.append(_proc(ims2))
        elif tta == "three_sizes":
            for s in (1.0, 0.9, 1.1):
                ims_s = [im.resize((int(im.width * s), int(im.height * s)), Image.BILINEAR) for im in ims_valid]
                embs.append(_proc(ims_s))
        else:
            embs.append(_proc(ims_valid))

        img_emb = torch.stack(embs, dim=0).mean(0)  # average TTA
    return img_emb, valid

# ---------- Text logits via MAX across prompts
def text_logits_max(img_emb):
    outs = []
    for lbl in labels:
        T = label_prompt_mats[lbl]     # [m, d]
        sims = img_emb @ T.T           # [n, m]
        mvals, _ = sims.max(dim=1)     # [n]
        outs.append(mvals.unsqueeze(1))
    return torch.cat(outs, dim=1)      # [n, C]

# ---------- Tip-Adapter-like prototype boost (no training)
def build_prototypes(K=8, per_class_min=1, seed=0):
    rng = np.random.RandomState(seed)
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    if not present:
        raise ValueError("No label_* columns in CSV.")

    prot_paths, prot_targets = [], []
    for ci, l in enumerate(labels):
        col = f"label_{l}"
        if col not in df.columns:
            continue
        pos_rows = df[df[col].astype(str).fillna("0").isin(["1", "1.0"])]
        pos_rows = pos_rows[pos_rows["path"].map(path_exists)]
        if len(pos_rows) < per_class_min:
            continue
        take = min(K, len(pos_rows))
        chosen = pos_rows.sample(take, random_state=seed)["path"].tolist()
        prot_paths.extend(chosen)
        prot_targets.extend([ci] * take)

    if not prot_paths:
        print("[WARN] No prototypes built.")
        return None, None

    P_emb, valid = embed_images(prot_paths, tta="center+shrink")
    valid_idx = [i for i, v in enumerate(valid) if v]
    P_emb = P_emb[valid_idx]
    prot_targets = [prot_targets[i] for i in valid_idx]

    C = len(labels)
    oh = torch.zeros((len(prot_targets), C), dtype=DTYPE, device=device)
    for i, ci in enumerate(prot_targets):
        oh[i, ci] = 1.0
    return P_emb, oh

def prototype_logits(img_emb, P_emb, oh, beta=80.0):
    sims = img_emb @ P_emb.T                     # [n, np] cosine sims
    knn = torch.softmax(beta * sims, dim=1)      # sharpen
    logits = knn @ oh                             # [n, C]
    return logits

# ---------- Per-class alpha (B)
DOC_CLUSTER = {"journal_banner", "doc_fullpage", "book_cover", "pure_text"}

def make_alpha_map(labels):
    # Lower alpha (more prototype) for doc-like; higher alpha for icon-like/simple
    base = {}
    for l in labels:
        if l in DOC_CLUSTER:
            base[l] = 0.60
        elif l in {"qr", "logo", "tiny"}:
            base[l] = 0.90
        else:
            base[l] = 0.80
    return base

# ---------- Doc-cluster tiny reranker (C)
def rerank_doc_cluster(logits):
    # logits: [n, C], mutate a copy
    idx = [labels.index(l) for l in DOC_CLUSTER if l in labels]
    if not idx:
        return logits
    out = logits.clone()
    sub = out[:, idx]  # [n, k]
    # Optional softmax/normalize inside the cluster; we just do argmax and nudge
    cluster_winner = sub.argmax(dim=1)  # [n]
    top1 = out.argmax(dim=1)
    for r in range(out.size(0)):
        if top1[r].item() in idx:
            winner_col = idx[cluster_winner[r].item()]
            # tiny nudge to stabilize within-cluster winner
            out[r, winner_col] += 1e-3
    return out

# ---------- Master scorer
class Scorer:
    def __init__(self,
                 use_max_text=True,
                 use_prototypes=True,
                 alpha=0.7,               # global fallback
                 alpha_map=None,          # dict(label -> alpha); overrides per class
                 beta=80.0,
                 tta_mode="center+shrink",
                 shots=8,
                 use_doc_rerank=True):
        self.use_max_text = use_max_text
        self.use_prototypes = use_prototypes
        self.alpha = alpha
        self.alpha_map = alpha_map or make_alpha_map(labels)
        self.beta = beta
        self.tta_mode = tta_mode
        self.use_doc_rerank = use_doc_rerank
        self.P = None
        self.OH = None
        if use_prototypes:
            self.P, self.OH = build_prototypes(K=shots, per_class_min=1, seed=0)
            if self.P is None:
                self.use_prototypes = False
                print("[INFO] Falling back to text-only (no prototypes).")

    def score_paths(self, paths):
        img_emb, valid = embed_images(paths, tta=self.tta_mode)
        if img_emb is None:
            return None, valid

        # text logits
        text_log = text_logits_max(img_emb) if self.use_max_text else text_logits_max(img_emb)

        # prototype logits
        if self.use_prototypes and self.P is not None:
            proto_log = prototype_logits(img_emb, self.P, self.OH, beta=self.beta)
            # mix per class
            alpha_vec = torch.tensor([self.alpha_map.get(l, self.alpha) for l in labels],
                                     device=text_log.device, dtype=text_log.dtype)
            final = alpha_vec * text_log + (1.0 - alpha_vec) * proto_log
        else:
            final = text_log

        if self.use_doc_rerank:
            final = rerank_doc_cluster(final)
        return final, valid

# ---------- Evaluation (image-level & per-class)
def evaluate(scorer, sample_size=64, seed=0, batch=16):
    rng = np.random.RandomState(seed)
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    assert present, "No label_* columns."

    # rows with at least one positive
    tmp = df[df[present].apply(lambda r: any(x in ["1","1.0",1,1.0,True] for x in r), axis=1)]
    if len(tmp) == 0:
        raise ValueError("No positive rows to evaluate.")
    n = min(sample_size, len(tmp))
    sample = tmp.sample(n, random_state=seed).reset_index(drop=True)

    paths = sample["path"].tolist()
    img_cnt = img_hit1 = img_hit5 = 0
    hit1 = {l: 0 for l in labels}
    hit5 = {l: 0 for l in labels}
    cnt  = {l: 0 for l in labels}

    for i in tqdm(range(0, len(paths), batch), desc="Scoring"):
        chunk = paths[i:i+batch]
        logits, valid = scorer.score_paths(chunk)
        if logits is None:
            continue
        logits_cpu = logits.detach().cpu()
        row_slice = sample.iloc[i:i+batch]

        k = 0
        for j, (_, r) in enumerate(row_slice.iterrows()):
            if not valid[j]:
                continue
            row_log = logits_cpu[k]; k += 1
            order = torch.argsort(row_log, descending=True).tolist()
            top1 = labels[order[0]]
            top5 = [labels[idx] for idx in order[:5]]
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,1.0,True]]

            # image-level
            img_cnt += 1
            img_hit1 += int(top1 in gt)
            img_hit5 += int(any(t in gt for t in top5))

            # per-class (original style)
            for g in gt:
                cnt[g] += 1
                if top1 == g:
                    hit1[g] += 1
                if g in top5:
                    hit5[g] += 1

    print("\n=== IMAGE-LEVEL METRICS (multi-label aware) ===")
    print(f"Hit@1: {img_hit1}/{img_cnt} ({img_hit1/img_cnt:.1%})")
    print(f"Hit@5: {img_hit5}/{img_cnt} ({img_hit5/img_cnt:.1%})")

    print("\n=== PER-CLASS (original style) ===")
    for l in labels:
        if cnt[l] > 0:
            print(f"{l:22s}: H@1={hit1[l]}/{cnt[l]}  H@5={hit5[l]}/{cnt[l]}")

# ---------- A) Export predictions
def evaluate_and_export(scorer, df_eval, out_csv="eval_predictions.csv", batch=32):
    rows = []
    labels_cols = [f"label_{l}" for l in labels]
    present = [c for c in labels_cols if c in df_eval.columns]
    df_eval = df_eval[df_eval[present].apply(lambda r: any(x in ["1","1.0",1,True] for x in r), axis=1)].reset_index(drop=True)

    paths = df_eval["path"].tolist()
    for i in tqdm(range(0, len(paths), batch), desc="Scoring (export)"):
        chunk = paths[i:i+batch]
        logits, valid = scorer.score_paths(chunk)
        if logits is None:
            continue
        L = logits.detach().cpu()
        for j, (_, r) in enumerate(df_eval.iloc[i:i+batch].iterrows()):
            if not valid[j]:
                continue
            row = L[j]
            order = torch.argsort(row, descending=True).tolist()
            top1_idx = order[0]
            top1 = labels[top1_idx]
            top5 = [labels[k] for k in order[:5]]
            gap = float(row[top1_idx] - row[order[1]]) if len(order) > 1 else float(row[top1_idx])
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,True]]
            rows.append({
                "path": r["path"],
                "gt": "|".join(gt),
                "pred_top1": top1,
                "top5": "|".join(top5),
                "gap": gap,
                "top1_score": float(row[top1_idx])
            })
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[INFO] Wrote {out_csv} with {len(rows)} rows")

# ---------- D) Simple grid search (returns top tuples)
def eval_image_level_hit1(df_eval, scorer):
    # quick image-level Hit@1 over df_eval
    labels_cols = [f"label_{l}" for l in labels]
    present = [c for c in labels_cols if c in df_eval.columns]
    df_eval = df_eval[df_eval[present].apply(lambda r: any(x in ["1","1.0",1,True] for x in r), axis=1)].reset_index(drop=True)
    paths = df_eval["path"].tolist()

    img_cnt = img_hit1 = 0
    for i in range(0, len(paths), 64):
        chunk = paths[i:i+64]
        logits, valid = scorer.score_paths(chunk)
        if logits is None:
            continue
        L = logits.detach().cpu()
        for j, (_, r) in enumerate(df_eval.iloc[i:i+64].iterrows()):
            if not valid[j]:
                continue
            order = torch.argsort(L[j], descending=True).tolist()
            pred = labels[order[0]]
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,True]]
            img_cnt += 1
            img_hit1 += int(pred in gt)
    return img_hit1 / max(1, img_cnt)

def grid_search(df_eval, alphas=(0.6,0.7,0.8,0.9), betas=(50,80,120), shots=(4,8,16), use_alpha_map=True):
    results = []
    alpha_map = make_alpha_map(labels) if use_alpha_map else None
    for a in alphas:
        for b in betas:
            for k in shots:
                s = Scorer(use_max_text=True, use_prototypes=True,
                           alpha=a, alpha_map=alpha_map, beta=b,
                           tta_mode="center+shrink", shots=k, use_doc_rerank=True)
                h1 = eval_image_level_hit1(df_eval, s)
                results.append((h1, a, b, k))
                print(f"[GRID] Hit@1={h1:.4f} | alpha={a} beta={b} shots={k}")
    results.sort(reverse=True, key=lambda x: x[0])
    print("\n[TOP GRID] (Hit@1, alpha, beta, shots)")
    for row in results[:5]:
        print("   ", row)
    return results

# ---------- RUN EXAMPLES

# 1) Fast dev eval on a 64-image sample
scorer = Scorer(
    use_max_text=True,
    use_prototypes=True,
    alpha=0.7,                     # global fallback
    alpha_map=make_alpha_map(labels),  # per-class alpha (B)
    beta=80.0,
    tta_mode="center+shrink",
    shots=8,
    use_doc_rerank=True            # (C)
)
evaluate(scorer, sample_size=64, seed=0, batch=16)

# 2) Export predictions for the full available set (A)
out_csv = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/invalid_filter/eval_preds_full.csv"
evaluate_and_export(scorer, df_eval=df, out_csv=out_csv, batch=32)

# 3) Grid search on the full available set (or subset) (D)
_ = grid_search(df_eval=df, alphas=(0.6,0.7,0.8,0.9), betas=(50,80,120), shots=(4,8,16), use_alpha_map=True)


Loading model from: /media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers/models--google--siglip-so400m-patch14-384/snapshots/9fdffc58afc957d1a03a25b10dba0329ab15c2a3
[INFO] Rows with existing files: 163
[INFO] Total prompts: 23


Scoring: 100%|██████████| 4/4 [00:05<00:00,  1.27s/it]



=== IMAGE-LEVEL METRICS (multi-label aware) ===
Hit@1: 55/64 (85.9%)
Hit@5: 64/64 (100.0%)

=== PER-CLASS (original style) ===
qr                    : H@1=3/3  H@5=3/3
logo                  : H@1=8/10  H@5=10/10
chart                 : H@1=2/2  H@5=2/2
journal_banner        : H@1=13/19  H@5=19/19
pure_text             : H@1=4/4  H@5=4/4
empty                 : H@1=7/7  H@5=7/7
ecg                   : H@1=1/1  H@5=1/1
profile               : H@1=1/2  H@5=2/2
gel_electrophoresis   : H@1=2/2  H@5=2/2
map                   : H@1=3/3  H@5=3/3
tiny                  : H@1=6/6  H@5=6/6
house                 : H@1=1/1  H@5=1/1
doc_fullpage          : H@1=4/4  H@5=4/4


Scoring (export): 100%|██████████| 5/5 [00:11<00:00,  2.22s/it]


[INFO] Wrote /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/invalid_filter/eval_preds_full.csv with 158 rows
[GRID] Hit@1=0.7278 | alpha=0.6 beta=50 shots=4
[GRID] Hit@1=0.8481 | alpha=0.6 beta=50 shots=8
[GRID] Hit@1=0.9177 | alpha=0.6 beta=50 shots=16
[GRID] Hit@1=0.7025 | alpha=0.6 beta=80 shots=4
[GRID] Hit@1=0.8291 | alpha=0.6 beta=80 shots=8
[GRID] Hit@1=0.9241 | alpha=0.6 beta=80 shots=16
[GRID] Hit@1=0.6962 | alpha=0.6 beta=120 shots=4
[GRID] Hit@1=0.8291 | alpha=0.6 beta=120 shots=8
[GRID] Hit@1=0.9241 | alpha=0.6 beta=120 shots=16
[GRID] Hit@1=0.7278 | alpha=0.7 beta=50 shots=4
[GRID] Hit@1=0.8481 | alpha=0.7 beta=50 shots=8
[GRID] Hit@1=0.9177 | alpha=0.7 beta=50 shots=16
[GRID] Hit@1=0.7025 | alpha=0.7 beta=80 shots=4
[GRID] Hit@1=0.8291 | alpha=0.7 beta=80 shots=8
[GRID] Hit@1=0.9241 | alpha=0.7 beta=80 shots=16
[GRID] Hit@1=0.6962 | alpha=0.7 beta=120 shots=4
[GRID] Hit@1=0.8291 | alpha=0.7 beta=120 shots=8
[GRID] Hit@1=0.9241 | alpha=0.7 beta=120 sh

In [7]:
# ==========================================
# v4.2: MAX-pool + TTA + Prototypes (shots=16)
#       + per-group alpha
#       + stronger doc-cluster reranker
#       + doc_vs_map guard
#       + export + grid search (kept)
# ==========================================

import os, json, math, random
from pathlib import Path
import numpy as np
import torch, pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm
from transformers import AutoProcessor, AutoModel

# ---------- Config
ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
DATA_CSV = ROOT / "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated_safe.csv"
SCHEMA_JSON = DATA_CSV.parent / "labels_schema_filtered_safe.json"

TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
try:
    SIGLIP_PATH = str(next(p for p in SIGLIP_DIR.iterdir() if p.is_dir()))
except StopIteration:
    SIGLIP_PATH = str(TRANSF_CACHE / "models--google--siglip-so400m-patch14-384")

# Offline caches
os.environ.setdefault("HF_HOME", str(ROOT / "hf"))
os.environ.setdefault("HF_HUB_CACHE", str(ROOT / "hf" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf" / "transformers"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# Device/dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
USE_AMP = (device == "cuda")

print("Loading model from:", SIGLIP_PATH)
processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# ---------- Data
labels = json.loads(Path(SCHEMA_JSON).read_text())
df = pd.read_csv(DATA_CSV, dtype=str, keep_default_na=False, low_memory=False)

def path_exists(p):
    try: return Path(p).exists()
    except: return False

if "path" not in df.columns:
    raise ValueError("CSV must have a 'path' column.")
df = df[df["path"].map(path_exists)]
print(f"[INFO] Rows with existing files: {len(df)}")

# ---------- Prompts (baseline detailed) + tight variants for MAX-pool
DETAILED_PROMPTS = {
    "qr": "a square black and white QR code",
    "logo": "a company or journal logo",
    "journal_banner": "a scientific journal article header",
    "pure_text": "a page with only text paragraphs",
    "book_cover": "a book cover with title text",
    "doc_fullpage": "a complete formatted document page",
    "empty": "an almost empty white page",
    "ecg": "an electrocardiogram waveform",
    "profile": "a portrait photo of a person",
    "gel_electrophoresis": "a gel with DNA or protein bands",
    "map": "a geographic map",
    "tiny": "a very small thumbnail image",
    "house": "a house or building",
    "illustration": "a diagram or illustration",
    "phylogenetic_tree": "a phylogenetic tree",
    "chart": "a data chart or graph",
}
VARIANTS = {
    "qr": ["a QR code"],
    "logo": ["a logo"],
    "journal_banner": ["journal header with title and authors"],
    "pure_text": ["text paragraphs only"],
    "doc_fullpage": ["full document page"],
    "gel_electrophoresis": ["gel electrophoresis bands"],
    "chart": ["data chart graph"],
}
def prompts_for_label(lbl):
    p = [DETAILED_PROMPTS.get(lbl, f"a {lbl.replace('_',' ')}")]
    p += VARIANTS.get(lbl, [])
    return p

# ---------- Text embeddings with MAX pooling at score time
with torch.no_grad():
    all_prompts, label_ranges, idx = [], {}, 0
    for lbl in labels:
        ps = prompts_for_label(lbl)
        label_ranges[lbl] = (idx, idx+len(ps))
        all_prompts.extend(ps)
        idx += len(ps)
    print(f"[INFO] Total prompts: {len(all_prompts)}")

    text_inputs = processor(text=all_prompts, return_tensors="pt", padding=True, truncation=True)
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            text_emb_all = model.get_text_features(**text_inputs)
    else:
        text_emb_all = model.get_text_features(**text_inputs)
    text_emb_all = torch.nn.functional.normalize(text_emb_all, dim=-1).to(DTYPE)
    del text_inputs

label_prompt_mats = {}
for lbl in labels:
    a, b = label_ranges[lbl]
    label_prompt_mats[lbl] = text_emb_all[a:b]

# ---------- Image embedder with light TTA
MIN_SIDE = 16
def load_rgb(path):
    try:
        im = Image.open(path).convert("RGB")
        w,h = im.size
        if min(w,h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w,h))
            im = im.resize((max(MIN_SIDE,int(w*scale)), max(MIN_SIDE,int(h*scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

def embed_images(paths, tta="center+shrink"):  # 'none' | 'center+shrink' | 'three_sizes'
    ims = [load_rgb(p) for p in paths]
    valid = [im is not None for im in ims]
    ims_valid = [im for im in ims if im is not None]
    if not ims_valid:
        return None, valid
    with torch.no_grad():
        def _proc(imgs):
            inputs = processor(images=imgs, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            if USE_AMP:
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    e = model.get_image_features(**inputs)
            else:
                e = model.get_image_features(**inputs)
            e = torch.nn.functional.normalize(e, dim=-1).to(DTYPE)
            return e
        embs = []
        if tta == "none":
            embs.append(_proc(ims_valid))
        elif tta == "center+shrink":
            embs.append(_proc(ims_valid))
            ims2 = [im.resize((int(im.width*0.9), int(im.height*0.9)), Image.BILINEAR) for im in ims_valid]
            embs.append(_proc(ims2))
        elif tta == "three_sizes":
            for s in (1.0, 0.9, 1.1):
                ims_s = [im.resize((int(im.width*s), int(im.height*s)), Image.BILINEAR) for im in ims_valid]
                embs.append(_proc(ims_s))
        else:
            embs.append(_proc(ims_valid))
        img_emb = torch.stack(embs, dim=0).mean(0)
    return img_emb, valid

def text_logits_max(img_emb):
    outs = []
    for lbl in labels:
        T = label_prompt_mats[lbl]   # [m, d]
        sims = img_emb @ T.T         # [n, m]
        mvals, _ = sims.max(dim=1)
        outs.append(mvals.unsqueeze(1))
    return torch.cat(outs, dim=1)    # [n, C]

# ---------- Prototypes (Tip-Adapter-like)
def build_prototypes(K=16, per_class_min=1, seed=0):
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    if not present: raise ValueError("No label_* columns in CSV.")
    prot_paths, prot_targets = [], []
    for ci, l in enumerate(labels):
        col = f"label_{l}"
        if col not in df.columns: continue
        pos_rows = df[df[col].astype(str).fillna("0").isin(["1","1.0"])]
        pos_rows = pos_rows[pos_rows["path"].map(path_exists)]
        if len(pos_rows) < per_class_min: continue
        take = min(K, len(pos_rows))
        chosen = pos_rows.sample(take, random_state=seed)["path"].tolist()
        prot_paths.extend(chosen); prot_targets.extend([ci]*take)
    if not prot_paths:
        print("[WARN] No prototypes built."); return None, None
    P_emb, valid = embed_images(prot_paths, tta="center+shrink")
    keep = [i for i,v in enumerate(valid) if v]
    P_emb = P_emb[keep]; prot_targets = [prot_targets[i] for i in keep]
    C = len(labels)
    oh = torch.zeros((len(prot_targets), C), dtype=DTYPE, device=device)
    for i, ci in enumerate(prot_targets): oh[i, ci] = 1.0
    return P_emb, oh

def prototype_logits(img_emb, P_emb, oh, beta=80.0):
    sims = img_emb @ P_emb.T
    knn = torch.softmax(beta * sims, dim=1)
    return knn @ oh

# ---------- Group alphas (doc/icon/other)
DOC_CLUSTER = {"journal_banner", "doc_fullpage", "book_cover", "pure_text"}
ICON_CLUSTER = {"qr","logo","tiny"}

def make_alpha_map(labels, alpha_doc=0.60, alpha_icon=0.90, alpha_other=0.80):
    mp = {}
    for l in labels:
        if l in DOC_CLUSTER: mp[l] = alpha_doc
        elif l in ICON_CLUSTER: mp[l] = alpha_icon
        else: mp[l] = alpha_other
    return mp

# ---------- Stronger doc reranker + doc-vs-map guard
def rerank_doc_cluster(logits):
    # reselect within doc cluster by softmax & reinsert winner score
    out = logits.clone()
    idx = [labels.index(l) for l in DOC_CLUSTER if l in labels]
    if not idx: return out
    sub = out[:, idx]                 # [n, k]
    probs = torch.softmax(sub * 10, dim=1)  # sharpen competition
    winner = probs.argmax(dim=1)      # [n]
    for r in range(out.size(0)):
        out[r, idx[winner[r]]] += 1e-3
    return out

def doc_map_guard(logits, margin=0.03, boost=0.02):
    # If 'map' is top1 but a doc-cluster score is within `margin`, tip to that doc
    out = logits.clone()
    if "map" not in labels: return out
    map_idx = labels.index("map")
    doc_idx = [labels.index(l) for l in DOC_CLUSTER if l in labels]
    dmax, didx = out[:, doc_idx].max(dim=1)  # best doc score + index inside doc_idx
    top1 = out.argmax(dim=1)
    for r in range(out.size(0)):
        if top1[r].item() == map_idx:
            s_map = out[r, map_idx].item()
            if dmax[r].item() >= s_map - margin:
                # tip scales slightly in favor of doc winner
                out[r, doc_idx[didx[r].item()]] += boost
    return out

# ---------- Master scorer
class Scorer:
    def __init__(self,
                 tta_mode="center+shrink",
                 alpha_doc=0.60, alpha_icon=0.90, alpha_other=0.80,
                 use_max_text=True,
                 use_prototypes=True,
                 beta=80.0,
                 shots=16,
                 use_doc_rerank=True,
                 use_doc_map_guard=True):
        self.tta_mode = tta_mode
        self.alpha_map = make_alpha_map(labels, alpha_doc, alpha_icon, alpha_other)
        self.use_max_text = use_max_text
        self.use_prototypes = use_prototypes
        self.beta = beta
        self.shots = shots
        self.use_doc_rerank = use_doc_rerank
        self.use_doc_map_guard = use_doc_map_guard
        self.P = None; self.OH = None
        if use_prototypes:
            self.P, self.OH = build_prototypes(K=shots, per_class_min=1, seed=0)
            if self.P is None:
                self.use_prototypes = False
                print("[INFO] Falling back to text-only (no prototypes).")

    def score_paths(self, paths):
        img_emb, valid = embed_images(paths, tta=self.tta_mode)
        if img_emb is None: return None, valid

        text_log = text_logits_max(img_emb) if self.use_max_text else text_logits_max(img_emb)
        if self.use_prototypes and self.P is not None:
            proto_log = prototype_logits(img_emb, self.P, self.OH, beta=self.beta)
            alpha_vec = torch.tensor([self.alpha_map[l] for l in labels],
                                     device=text_log.device, dtype=text_log.dtype)
            final = alpha_vec * text_log + (1.0 - alpha_vec) * proto_log
        else:
            final = text_log

        if self.use_doc_rerank:
            final = rerank_doc_cluster(final)
        if self.use_doc_map_guard:
            final = doc_map_guard(final, margin=0.03, boost=0.02)
        return final, valid

# ---------- Eval + Export + Grid (unchanged from v4.1 except defaults)
def evaluate(scorer, sample_size=64, seed=0, batch=16):
    label_cols = [f"label_{l}" for l in labels]
    present = [c for c in label_cols if c in df.columns]
    tmp = df[df[present].apply(lambda r: any(x in ["1","1.0",1,1.0,True] for x in r), axis=1)]
    if len(tmp) == 0: raise ValueError("No positive rows to evaluate.")
    n = min(sample_size, len(tmp))
    sample = tmp.sample(n, random_state=seed).reset_index(drop=True)

    paths = sample["path"].tolist()
    img_cnt = img_hit1 = img_hit5 = 0
    hit1 = {l: 0 for l in labels}; hit5 = {l: 0 for l in labels}; cnt = {l: 0 for l in labels}

    for i in tqdm(range(0, len(paths), batch), desc="Scoring"):
        chunk = paths[i:i+batch]
        logits, valid = scorer.score_paths(chunk)
        if logits is None: continue
        L = logits.detach().cpu()
        row_slice = sample.iloc[i:i+batch]
        k = 0
        for j, (_, r) in enumerate(row_slice.iterrows()):
            if not valid[j]: continue
            row = L[k]; k += 1
            order = torch.argsort(row, descending=True).tolist()
            top1 = labels[order[0]]
            top5 = [labels[idx] for idx in order[:5]]
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,1.0,True]]
            img_cnt += 1
            img_hit1 += int(top1 in gt)
            img_hit5 += int(any(t in gt for t in top5))
            for g in gt:
                cnt[g] += 1
                if top1 == g: hit1[g] += 1
                if g in top5: hit5[g] += 1

    print("\n=== IMAGE-LEVEL METRICS (multi-label aware) ===")
    print(f"Hit@1: {img_hit1}/{img_cnt} ({img_hit1/img_cnt:.1%})")
    print(f"Hit@5: {img_hit5}/{img_cnt} ({img_hit5/img_cnt:.1%})")

    print("\n=== PER-CLASS (original style) ===")
    for l in labels:
        if cnt[l] > 0:
            print(f"{l:22s}: H@1={hit1[l]}/{cnt[l]}  H@5={hit5[l]}/{cnt[l]}")

def evaluate_and_export(scorer, df_eval, out_csv="eval_preds_full.csv", batch=32):
    rows = []
    labels_cols = [f"label_{l}" for l in labels]
    present = [c for c in labels_cols if c in df_eval.columns]
    df_eval = df_eval[df_eval[present].apply(lambda r: any(x in ["1","1.0",1,True] for x in r), axis=1)].reset_index(drop=True)
    paths = df_eval["path"].tolist()
    for i in tqdm(range(0, len(paths), batch), desc="Scoring (export)"):
        chunk = paths[i:i+batch]
        logits, valid = scorer.score_paths(chunk)
        if logits is None: continue
        L = logits.detach().cpu()
        for j, (_, r) in enumerate(df_eval.iloc[i:i+batch].iterrows()):
            if not valid[j]: continue
            row = L[j]
            order = torch.argsort(row, descending=True).tolist()
            top1_idx = order[0]; top1 = labels[top1_idx]
            top5 = [labels[k] for k in order[:5]]
            gap = float(row[top1_idx] - row[order[1]]) if len(order) > 1 else float(row[top1_idx])
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,True]]
            rows.append({
                "path": r["path"],
                "gt": "|".join(gt),
                "pred_top1": top1,
                "top5": "|".join(top5),
                "gap": gap,
                "top1_score": float(row[top1_idx])
            })
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[INFO] Wrote {Path(out_csv).resolve()} with {len(rows)} rows")

def eval_image_level_hit1(df_eval, scorer):
    labels_cols = [f"label_{l}" for l in labels]
    present = [c for c in labels_cols if c in df_eval.columns]
    df_eval = df_eval[df_eval[present].apply(lambda r: any(x in ["1","1.0",1,True] for x in r), axis=1)].reset_index(drop=True)
    paths = df_eval["path"].tolist()
    img_cnt = img_hit1 = 0
    for i in range(0, len(paths), 64):
        chunk = paths[i:i+64]
        logits, valid = scorer.score_paths(chunk)
        if logits is None: continue
        L = logits.detach().cpu()
        for j, (_, r) in enumerate(df_eval.iloc[i:i+64].iterrows()):
            if not valid[j]: continue
            order = torch.argsort(L[j], descending=True).tolist()
            pred = labels[order[0]]
            gt = [l for l in labels if r.get(f"label_{l}", "0") in ["1","1.0",1,True]]
            img_cnt += 1; img_hit1 += int(pred in gt)
    return img_hit1 / max(1, img_cnt)

def grid_search(df_eval,
                alpha_doc_vals=(0.55,0.60,0.65),
                alpha_icon_vals=(0.85,0.90,0.95),
                alpha_other_vals=(0.75,0.80,0.85),
                betas=(80,120),
                shots=(16,),   # best from your sweep
                tta="center+shrink"):
    results = []
    for a_doc in alpha_doc_vals:
        for a_icon in alpha_icon_vals:
            for a_oth in alpha_other_vals:
                for b in betas:
                    for k in shots:
                        s = Scorer(tta_mode=tta,
                                   alpha_doc=a_doc, alpha_icon=a_icon, alpha_other=a_oth,
                                   use_max_text=True, use_prototypes=True,
                                   beta=b, shots=k, use_doc_rerank=True, use_doc_map_guard=True)
                        h1 = eval_image_level_hit1(df_eval, s)
                        results.append((h1, a_doc, a_icon, a_oth, b, k))
                        print(f"[GRID] Hit@1={h1:.4f} | a_doc={a_doc} a_icon={a_icon} a_oth={a_oth} beta={b} shots={k}")
    results.sort(reverse=True, key=lambda x: x[0])
    print("\n[TOP GRID] (Hit@1, a_doc, a_icon, a_oth, beta, shots)")
    for row in results[:5]:
        print("   ", row)
    return results

# ---------- RUN (defaults chosen from your top grid)
scorer = Scorer(
    tta_mode="center+shrink",
    alpha_doc=0.60, alpha_icon=0.90, alpha_other=0.80,
    use_max_text=True,
    use_prototypes=True,
    beta=80.0,           # as good as 120, cheaper
    shots=16,            # best according to your grid
    use_doc_rerank=True,
    use_doc_map_guard=True
)

# Quick sample eval
evaluate(scorer, sample_size=64, seed=0, batch=16)

# Full export (optional)
evaluate_and_export(scorer, df_eval=df, out_csv=str(DATA_CSV.parent / "eval_preds_full_v42.csv"), batch=32)

# Focused grid over group-alphas (optional)
_ = grid_search(df_eval=df)

Loading model from: /media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers/models--google--siglip-so400m-patch14-384/snapshots/9fdffc58afc957d1a03a25b10dba0329ab15c2a3
[INFO] Rows with existing files: 163
[INFO] Total prompts: 23


Scoring: 100%|██████████| 4/4 [00:04<00:00,  1.07s/it]



=== IMAGE-LEVEL METRICS (multi-label aware) ===
Hit@1: 61/64 (95.3%)
Hit@5: 64/64 (100.0%)

=== PER-CLASS (original style) ===
qr                    : H@1=3/3  H@5=3/3
logo                  : H@1=8/10  H@5=10/10
chart                 : H@1=2/2  H@5=2/2
journal_banner        : H@1=19/19  H@5=19/19
pure_text             : H@1=4/4  H@5=4/4
empty                 : H@1=7/7  H@5=7/7
ecg                   : H@1=1/1  H@5=1/1
profile               : H@1=1/2  H@5=2/2
gel_electrophoresis   : H@1=2/2  H@5=2/2
map                   : H@1=3/3  H@5=3/3
tiny                  : H@1=6/6  H@5=6/6
house                 : H@1=1/1  H@5=1/1
doc_fullpage          : H@1=4/4  H@5=4/4


Scoring (export): 100%|██████████| 5/5 [00:12<00:00,  2.47s/it]


[INFO] Wrote /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/invalid_filter/eval_preds_full_v42.csv with 158 rows
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.75 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.75 beta=120 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.8 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.8 beta=120 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.85 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.85 a_oth=0.85 beta=120 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.75 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.75 beta=120 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.8 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.8 beta=120 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.85 beta=80 shots=16
[GRID] Hit@1=0.9241 | a_doc=0.55 a_icon=0.9 a_oth=0.85 beta=120 sh

1) Setup & utilities (reuses your SIGLIP load; adds embedding cache helpers)

In [1]:
# Cell 1 — Setup & utilities

import os, json, math, time, gc, random
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

import torch
from transformers import AutoProcessor, AutoModel

# ---------- Paths (edit ROOT if needed)
ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
DATA_CSV = ROOT / "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated_safe.csv"
SCHEMA_JSON = DATA_CSV.parent / "labels_schema_filtered_safe.json"

# Target corpus (49k images)
BIG_IMG_DIR = ROOT / "kaggle/working_v2/rag_knowledge_base/images"

# Offline caches
os.environ.setdefault("HF_HOME", str(ROOT / "hf"))
os.environ.setdefault("HF_HUB_CACHE", str(ROOT / "hf" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf" / "transformers"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# ---------- Model cache location
TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
if SIGLIP_DIR.exists():
    snaps = [p for p in SIGLIP_DIR.iterdir() if p.is_dir()]
    assert len(snaps) >= 1, f"No snapshots in {SIGLIP_DIR}"
    SIGLIP_PATH = str(snaps[0])
else:
    SIGLIP_PATH = str(TRANSF_CACHE / "models--google--siglip-so400m-patch14-384")

# ---------- Device / dtype
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if device == "cuda" else torch.float32
USE_AMP = (device == "cuda")

print("Loading model from:", SIGLIP_PATH)
processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# ---------- Labels & data
labels: List[str] = json.loads(Path(SCHEMA_JSON).read_text())
C = len(labels)

# ---------- IO helpers
def path_exists(p): 
    try:
        return Path(p).exists()
    except:
        return False

def list_images(root: Path, exts={".png",".jpg",".jpeg",".bmp",".gif",".tif",".tiff"}):
    root = Path(root)
    if not root.exists():
        return []
    out = []
    for p in root.rglob("*"):
        if p.suffix.lower() in exts:
            out.append(str(p))
    return out

# ---------- Embedding functions
MIN_SIDE = 16

def load_rgb(path: str):
    try:
        im = Image.open(path).convert("RGB")
        w,h = im.size
        if min(w,h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w,h))
            im = im.resize((max(MIN_SIDE,int(w*scale)), max(MIN_SIDE,int(h*scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

def embed_batch(imgs: List[Image.Image]) -> torch.Tensor:
    inputs = processor(images=imgs, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if USE_AMP:
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            e = model.get_image_features(**inputs)
    else:
        e = model.get_image_features(**inputs)
    e = torch.nn.functional.normalize(e, dim=-1)
    e = e.to(DTYPE)
    return e

def embed_paths(paths: List[str], batch_size=64, tta=False) -> Tuple[np.ndarray, List[bool]]:
    embs = []
    valid_mask = []
    for i in tqdm(range(0, len(paths), batch_size), desc="Embedding"):
        chunk = paths[i:i+batch_size]
        imgs = [load_rgb(p) for p in chunk]
        valids = [im is not None for im in imgs]
        valid_mask.extend(valids)
        imgs_valid = [im for im in imgs if im is not None]
        if len(imgs_valid)==0:
            continue

        # optional simple TTA: average normal and 0.9x resize
        if not tta:
            e = embed_batch(imgs_valid)
        else:
            e1 = embed_batch(imgs_valid)
            imgs2 = [im.resize((int(im.width*0.9),int(im.height*0.9)), Image.BILINEAR) for im in imgs_valid]
            e2 = embed_batch(imgs2)
            e = (e1 + e2) / 2.0

        embs.append(e.detach().cpu().numpy())

        # free a bit
        del imgs_valid
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

    if len(embs)==0:
        return np.zeros((0, model.config.projection_dim), dtype=np.float32), valid_mask
    E = np.concatenate(embs, axis=0).astype(np.float32)
    return E, valid_mask

print(f"[INFO] Loaded labels ({len(labels)}): {labels[:8]}{' ...' if len(labels)>8 else ''}")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading model from: /media/pc1/Ubuntu/Extend_Data/ngoc/hf/transformers/models--google--siglip-so400m-patch14-384/snapshots/9fdffc58afc957d1a03a25b10dba0329ab15c2a3
[INFO] Loaded labels (16): ['qr', 'logo', 'chart', 'journal_banner', 'pure_text', 'book_cover', 'empty', 'ecg'] ...


2) Cache embeddings for the labeled set (train/val) and for the 49k pool

In [ ]:
# === Cell 2-R: Cache SigLIP image embeddings (auto batch, sharded, fp16) ===
import os, math, gc, csv, time, json
from pathlib import Path
from typing import List, Tuple
import numpy as np
import torch
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, AutoModel

# --------- Paths (edit the IMAGES_DIR to your big corpus) ----------
ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
IMAGES_DIR = ROOT / "kaggle/working_v2/rag_knowledge_base/images"   # <-- 49,216 imgs
CACHE_DIR  = ROOT / "kaggle/working_v2/emb_cache_siglip"            # outputs here
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Offline & HF caches (kept from your setup)
os.environ.setdefault("HF_HOME", str(ROOT / "hf"))
os.environ.setdefault("HF_HUB_CACHE", str(ROOT / "hf" / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(ROOT / "hf" / "transformers"))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

# --------- Model ----------
TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
if SIGLIP_DIR.exists():
    snaps = [p for p in SIGLIP_DIR.iterdir() if p.is_dir()]
    assert snaps, f"No snapshots under {SIGLIP_DIR}"
    SIGLIP_PATH = str(snaps[0])
else:
    SIGLIP_PATH = str(TRANSF_CACHE / "models--google--siglip-so400m-patch14-384")

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if device == "cuda" else torch.float32
print("Loading model from:", SIGLIP_PATH)

processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# optional: allow TF32 to improve speed on Ampere
torch.backends.cuda.matmul.allow_tf32 = True

# --------- Image list ----------
def list_images(folder: Path) -> List[str]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    paths = []
    for p in folder.rglob("*"):
        if p.suffix.lower() in exts:
            paths.append(str(p))
    return sorted(paths)

all_paths = list_images(IMAGES_DIR)
N = len(all_paths)
print(f"[INFO] Found {N} images under {IMAGES_DIR}")

# --------- Robust loader ----------
MIN_SIDE = 16
def load_rgb(path: str):
    try:
        im = Image.open(path).convert("RGB")
        w,h = im.size
        if min(w,h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w,h))
            im = im.resize((max(MIN_SIDE,int(w*scale)), max(MIN_SIDE,int(h*scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

def collate(images):
    # HuggingFace processors can take a list of PIL Images directly.
    return processor(images=images, return_tensors="pt")

# --------- Empirical VRAM probe (answers your question) ----------
def mem_bytes():
    if device == "cuda":
        free, total = torch.cuda.mem_get_info()
        used = total - free
        return used, total
    return (0, 0)

@torch.no_grad()
def vram_per_image_probe(test_paths: List[str], max_try_bs=256, safety_gb=1.0) -> Tuple[int, float]:
    """
    Returns (recommended_batch, per_image_MB).
    Strategy: measure peak allocated for a few batch sizes, fit linear slope (MB/img),
    then pick batch so that we stay below ~85% of available VRAM minus safety margin.
    """
    if device != "cuda":
        return 1, 0.0

    # warm-up
    imgs = [load_rgb(p) for p in test_paths[:1] if load_rgb(p) is not None]
    if not imgs: return 1, 0.0
    _ = model.get_image_features(pixel_values=collate(imgs)["pixel_values"].to(device, dtype=DTYPE))

    torch.cuda.empty_cache(); gc.collect()
    points = []
    for bs in [1, 2, 4, 8, 16, 32, 64]:
        if bs > max_try_bs: break
        batch_imgs = []
        i = 0
        while len(batch_imgs) < bs and i < len(test_paths):
            im = load_rgb(test_paths[i]); i += 1
            if im is not None: batch_imgs.append(im)
        if len(batch_imgs) < bs: break

        torch.cuda.reset_peak_memory_stats()
        used0, total = mem_bytes()
        try:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                inputs = collate(batch_imgs)
                pix = inputs["pixel_values"].to(device, dtype=DTYPE, non_blocking=True)
                _ = model.get_image_features(pixel_values=pix)
            torch.cuda.synchronize()
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                break
            else:
                raise
        peak = torch.cuda.max_memory_allocated()
        points.append((bs, peak))
        del inputs, pix, batch_imgs, _
        torch.cuda.empty_cache(); gc.collect()

    if len(points) < 2:
        return 1, 0.0

    # linear fit: mem = a*bs + b
    bs_arr = np.array([p[0] for p in points], dtype=float)
    mem_arr = np.array([p[1] for p in points], dtype=float)  # bytes
    A = np.vstack([bs_arr, np.ones_like(bs_arr)]).T
    a, b = np.linalg.lstsq(A, mem_arr, rcond=None)[0]  # bytes per image, base bytes
    per_img_mb = a / (1024**2)

    # choose batch to use ~85% of free VRAM (minus safety)
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    target = 0.85 * total_bytes - safety_gb * (1024**3)  # bytes
    rec_bs = max(1, int((target - b) / max(a, 1)))       # avoid div0
    print(f"[VRAM] per-image ~{per_img_mb:.2f} MB; base ~{b/(1024**2):.0f} MB; total={total_bytes/(1024**3):.1f} GB")
    print(f"[VRAM] recommended batch ~= {rec_bs}")
    return rec_bs, per_img_mb

probe_bs, per_img_mb = vram_per_image_probe(all_paths[:512])  # probe on a small slice
DEFAULT_BS = max(8, min(128, probe_bs)) if device == "cuda" else 32
print(f"[INFO] Using batch_size={DEFAULT_BS}")

# --------- Sharded memmap sink ----------
DIM = 1152  # SigLIP so400m embedding width (see HF issues with 1152 hidden & 729 pos)  # noqa
EMB_DTYPE = np.float16

memmap_path = CACHE_DIR / "siglip_img_fp16.memmap"
index_csv   = CACHE_DIR / "siglip_img_index.csv"

# If resuming, don't overwrite existing memmap; size it to full N
emb_mmap = np.memmap(memmap_path, mode="w+", dtype=EMB_DTYPE, shape=(N, DIM))

with open(index_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["row", "path"])

# --------- Main loop (streaming) ----------
@torch.no_grad()
def embed_folder(paths: List[str], batch_size: int = DEFAULT_BS):
    ok = 0
    t0 = time.time()
    row = 0
    with open(index_csv, "a", newline="") as f:
        w = csv.writer(f)
        for i in range(0, len(paths), batch_size):
            chunk = paths[i:i+batch_size]
            imgs = [load_rgb(p) for p in chunk]
            keep = [(p, im) for p, im in zip(chunk, imgs) if im is not None]
            if not keep: continue

            ims = [im for _, im in keep]
            inputs = collate(ims)
            pix = inputs["pixel_values"].to(device, dtype=DTYPE, non_blocking=True)

            if device == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    feats = model.get_image_features(pixel_values=pix)
            else:
                feats = model.get_image_features(pixel_values=pix)

            feats = torch.nn.functional.normalize(feats, dim=-1)
            feats = feats.to(torch.float16).cpu().numpy()  # compact

            # write into memmap and index
            for j, (p, _) in enumerate(keep):
                emb_mmap[row, :] = feats[j]
                w.writerow([row, p])
                row += 1
            ok += len(keep)

            # explicit cleanup between steps
            del inputs, pix, feats, ims, imgs, keep
            if device == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

            if (i // batch_size) % 50 == 0:
                done = min(i + batch_size, len(paths))
                rate = ok / max(1e-9, (time.time() - t0))
                print(f"[{done}/{len(paths)}] cached={ok}, ~{rate:.1f} img/s")

    print(f"[DONE] wrote {ok} / {len(paths)} embeddings to {memmap_path}")
    return ok

cached = embed_folder(all_paths, batch_size=DEFAULT_BS)

# flush memmap to disk
emb_mmap.flush()
print(f"[INFO] Memmap size: {memmap_path.stat().st_size/1e6:.1f} MB ; Index rows={cached}")

[INFO] Labeled rows with existing files: 163


Embedding:   0%|          | 0/3 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 384.00 MiB. GPU 0 has a total capacity of 23.68 GiB of which 328.06 MiB is free. Including non-PyTorch memory, this process has 22.98 GiB memory in use. Of the allocated memory 22.53 GiB is allocated by PyTorch, and 150.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
# === Cell 3R — Build & SAVE: text-bank, prototypes (from labeled images), OvR head, thresholds ===
import os, json, math, gc, random
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import torch
from PIL import Image, UnidentifiedImageError
from transformers import AutoProcessor, AutoModel

ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
CACHE_DIR  = ROOT / "kaggle/working_v2/emb_cache_siglip"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LABELED_CSV = ROOT / "kaggle/working/rag_knowledge_base/invalid_filter/dataset_updated_safe.csv"

# ----- model (small batches; we embed only ~hundreds of images here)
TRANSF_CACHE = ROOT / "hf" / "transformers"
SIGLIP_DIR = TRANSF_CACHE / "models--google--siglip-so400m-patch14-384" / "snapshots"
SIGLIP_PATH = str(next((p for p in SIGLIP_DIR.iterdir() if p.is_dir()), TRANSF_CACHE / "models--google--siglip-so400m-patch14-384"))

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if device == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(SIGLIP_PATH, local_files_only=True)
model = AutoModel.from_pretrained(SIGLIP_PATH, local_files_only=True, torch_dtype=DTYPE).to(device).eval()

# ----- labels & prompts (same set you used in v4.2)
DETAILED_PROMPTS = {
    "qr": "a square black and white QR code",
    "logo": "a company or journal logo",
    "journal_banner": "a scientific journal article header",
    "pure_text": "a page with only text paragraphs",
    "book_cover": "a book cover with title text",
    "doc_fullpage": "a complete formatted document page",
    "empty": "an almost empty white page",
    "ecg": "an electrocardiogram waveform",
    "profile": "a portrait photo of a person",
    "gel_electrophoresis": "a gel with DNA or protein bands",
    "map": "a geographic map",
    "tiny": "a very small thumbnail image",
    "house": "a house or building",
    "illustration": "a diagram or illustration",
    "phylogenetic_tree": "a phylogenetic tree",
    "chart": "a data chart or graph",
}
VARIANTS = {
    "qr": ["a QR code"],
    "logo": ["a logo"],
    "journal_banner": ["journal header with title and authors"],
    "pure_text": ["text paragraphs only"],
    "doc_fullpage": ["full document page"],
    "gel_electrophoresis": ["gel electrophoresis bands"],
    "chart": ["data chart graph"],
}
labels = list(DETAILED_PROMPTS.keys())

def prompts_for_label(lbl):
    p = [DETAILED_PROMPTS.get(lbl, f"a {lbl.replace('_',' ')}")]
    p += VARIANTS.get(lbl, [])
    return p

# ----- tiny image helpers
MIN_SIDE = 16
def load_rgb(path: str):
    try:
        im = Image.open(path).convert("RGB")
        w,h = im.size
        if min(w,h) < MIN_SIDE:
            scale = MIN_SIDE / max(1, min(w,h))
            im = im.resize((max(MIN_SIDE,int(w*scale)), max(MIN_SIDE,int(h*scale))), Image.BILINEAR)
        return im
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

@torch.no_grad()
def embed_pils(imgs: List[Image.Image]) -> torch.Tensor:
    if not imgs: return torch.empty(0, 1152)
    inputs = processor(images=imgs, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    if device == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            e = model.get_image_features(**inputs)
    else:
        e = model.get_image_features(**inputs)
    return torch.nn.functional.normalize(e, dim=-1).to(torch.float32)

# ----- 1) TEXT BANK (max-pool over variants at score-time)
with torch.no_grad():
    all_prompts, label_ranges, idx = [], {}, 0
    for lbl in labels:
        ps = prompts_for_label(lbl)
        label_ranges[lbl] = (idx, idx+len(ps))
        all_prompts.extend(ps); idx += len(ps)

    tin = processor(text=all_prompts, return_tensors="pt", padding=True, truncation=True)
    tin = {k: v.to(device) for k, v in tin.items()}
    if device == "cuda":
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            text_emb_all = model.get_text_features(**tin)
    else:
        text_emb_all = model.get_text_features(**tin)
    text_emb_all = torch.nn.functional.normalize(text_emb_all, dim=-1).to(torch.float32).cpu()

np.savez(CACHE_DIR / "text_bank_siglip_v1.npz",
         emb=text_emb_all.numpy(),
         labels=np.array(labels, dtype=object),
         starts=np.array([label_ranges[l][0] for l in labels], dtype=np.int32),
         ends=np.array([label_ranges[l][1] for l in labels], dtype=np.int32))
print("[SAVE] text_bank_siglip_v1.npz")

# ----- 2) PROTOTYPES (embed labeled POS examples directly; no cache overlap needed)
df_lab = pd.read_csv(LABELED_CSV, dtype=str, keep_default_na=False, low_memory=False)
shots = 16
P_list, targets = [], []
for ci, lbl in enumerate(labels):
    col = f"label_{lbl}"
    if col not in df_lab.columns: continue
    pos = df_lab[df_lab[col].isin(["1","1.0",1,True])]["path"].tolist()
    random.Random(0).shuffle(pos)
    take = min(shots, len(pos))
    pos = pos[:take]
    # embed in small batches
    bs = 64
    for i in range(0, len(pos), bs):
        imgs = [load_rgb(p) for p in pos[i:i+bs]]
        imgs = [im for im in imgs if im is not None]
        if not imgs: continue
        P_list.append(embed_pils(imgs).cpu().numpy())
        targets.extend([ci]*len(imgs))

if P_list:
    P = np.concatenate(P_list, axis=0).astype(np.float32)  # [nP, 1152]
    OH = np.zeros((len(targets), len(labels)), dtype=np.float32)
    for i, ci in enumerate(targets): OH[i, ci] = 1.0
    np.savez(CACHE_DIR / "protos_siglip_v1.npz",
             P=P, OH=OH, labels=np.array(labels, dtype=object))
    print("[SAVE] protos_siglip_v1.npz  -> prototypes:", P.shape[0])
else:
    print("[WARN] No prototypes could be embedded (check labeled CSV image paths).")

# ----- 3) Train OvR logistic head (optional) on ALL labeled embeddings
# (pos+neg), so thresholds are on calibrated probabilities.
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import SGDClassifier
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.pipeline import make_pipeline
    from joblib import dump
    SK_OK = True
except Exception:
    SK_OK = False

def embed_labeled_rows(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    X_list, Y_list, used = [], [], []
    bs = 64
    rows = df.to_dict("records")
    for i in range(0, len(rows), bs):
        chunk = rows[i:i+bs]
        imgs, keep = [], []
        for r in chunk:
            p = r["path"]
            im = load_rgb(p)
            if im is not None:
                imgs.append(im); keep.append(r)
        if not imgs: continue
        E = embed_pils(imgs).cpu().numpy().astype(np.float32)
        X_list.append(E)
        for r in keep:
            Y_list.append([int(str(r.get(f"label_{lbl}", "0")) in ["1","1.0"]) for lbl in labels])
            used.append(r["path"])
    if not X_list:
        return np.zeros((0,1152), np.float32), np.zeros((0,len(labels)), np.int32), []
    return np.concatenate(X_list, axis=0), np.array(Y_list, np.int32), used

X_lab, Y_lab, used_paths = embed_labeled_rows(df_lab)
print(f"[INFO] Labeled embed count: {X_lab.shape[0]}")

ovr_saved = None
if SK_OK and X_lab.shape[0] > 0:
    clf = make_pipeline(
        StandardScaler(with_mean=False),
        OneVsRestClassifier(SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=3000, n_jobs=-1, random_state=0))
    )
    clf.fit(X_lab, Y_lab)
    dump({"labels": labels, "pipeline": clf}, CACHE_DIR / "ovr_logreg_v1.joblib")
    print("[SAVE] ovr_logreg_v1.joblib")
    ovr_saved = CACHE_DIR / "ovr_logreg_v1.joblib"

# ----- 4) Calibrate thresholds for the PROTO scorer from labeled set
# We score the labeled embeddings with the proto+text scorer and take a conservative quantile.
def load_text_bank():
    z = np.load(CACHE_DIR / "text_bank_siglip_v1.npz", allow_pickle=True)
    return z["emb"].astype(np.float32), list(z["labels"]), z["starts"], z["ends"]

def make_proto_text_scorer(text_emb_all, labels, starts, ends, P=None, OH=None, mix=(0.55,0.85,0.75), beta=80.0):
    ranges = {lbl: (int(starts[i]), int(ends[i])) for i,lbl in enumerate(labels)}
    # doc rerank + guard (same as before, light)
    DOC = {"journal_banner","doc_fullpage","pure_text","book_cover"}
    def score(X: np.ndarray) -> np.ndarray:
        X = X.astype(np.float32)                           # [B,1152]
        img = torch.from_numpy(X)
        sims = []
        for lbl in labels:
            a,b = ranges[lbl]
            T = torch.from_numpy(text_emb_all[a:b])
            s = img @ T.T
            sims.append(s.max(dim=1).values.unsqueeze(1))
        text_logits = torch.cat(sims, dim=1)
        if P is None:
            out = text_logits
        else:
            P_t = torch.from_numpy(P.astype(np.float32))
            OH_t= torch.from_numpy(OH.astype(np.float32))
            knn = torch.softmax(beta * (img @ P_t.T), dim=1)
            proto_logits = knn @ OH_t
            # per-group alpha
            a_doc, a_icon, a_oth = mix
            ICON = {"logo","qr","tiny","empty"}
            A = []
            for lbl in labels:
                if lbl in DOC:   A.append(a_doc)
                elif lbl in ICON:A.append(a_icon)
                else:            A.append(a_oth)
            A = torch.tensor(A, dtype=torch.float32).view(1,-1)
            out = A * text_logits + (1.0 - A) * proto_logits
        # doc rerank: tiny bump to doc-cluster winner
        idx = [i for i,l in enumerate(labels) if l in DOC]
        if idx:
            sub = out[:, idx]
            win = sub.softmax(dim=1).argmax(dim=1)
            for r in range(out.size(0)):
                out[r, idx[win[r].item()]] += 1e-3
        return out.numpy()
    return score

# load text bank & prototypes (the ones we just saved)
text_emb_all, LBL, starts, ends = load_text_bank()
proto = np.load(CACHE_DIR / "protos_siglip_v1.npz", allow_pickle=True) if (CACHE_DIR / "protos_siglip_v1.npz").exists() else None
P = proto["P"] if proto is not None else None
OH= proto["OH"] if proto is not None else None

proto_scorer = make_proto_text_scorer(text_emb_all, LBL, starts, ends, P=P, OH=OH, mix=(0.55,0.85,0.75), beta=80.0)

# score labeled embeds
if X_lab.shape[0] > 0:
    S_lab = proto_scorer(X_lab)     # [N_lab, C]
    # per-label thresholds: 15th percentile of positive scores (conservative)
    TH = {}
    for ci,lbl in enumerate(labels):
        pos = S_lab[Y_lab[:,ci]==1, ci]
        if pos.size > 0:
            TH[lbl] = float(np.quantile(pos, 0.15))
        else:
            TH[lbl] = 0.18  # fallback
    with open(CACHE_DIR / "thresholds_proto_v1.json", "w") as f:
        json.dump(TH, f, indent=2)
    print("[SAVE] thresholds_proto_v1.json")
else:
    print("[WARN] Could not calibrate thresholds (no labeled embeddings).")

# free GPU ASAP
try:
    model.to("cpu"); del model, processor
except Exception:
    pass
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("[READY] Saved text bank, prototypes, (optional) OvR, and thresholds.")

[SAVE] text_bank_siglip_v1.npz
[SAVE] protos_siglip_v1.npz  -> prototypes: 101
[INFO] Labeled embed count: 163
[SAVE] ovr_logreg_v1.joblib
[SAVE] thresholds_proto_v1.json
[READY] Saved text bank, prototypes, (optional) OvR, and thresholds.


In [7]:
# === Cell 4R — Score cached embeddings and FILTER ===
import os, json, gc
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
CACHE_DIR  = ROOT / "kaggle/working_v2/emb_cache_siglip"
memmap_path = CACHE_DIR / "siglip_img_fp16.memmap"
index_csv   = CACHE_DIR / "siglip_img_index.csv"

index_df = pd.read_csv(index_csv)
N = len(index_df)
DIM = 1152
E = np.memmap(memmap_path, mode="r", dtype=np.float16, shape=(N, DIM))

# load text bank, prototypes, thresholds
tb = np.load(CACHE_DIR / "text_bank_siglip_v1.npz", allow_pickle=True)
labels = list(tb["labels"])
text_emb_all = tb["emb"].astype(np.float32)
starts, ends = tb["starts"], tb["ends"]

proto = np.load(CACHE_DIR / "protos_siglip_v1.npz", allow_pickle=True) if (CACHE_DIR / "protos_siglip_v1.npz").exists() else None
P  = proto["P"].astype(np.float32) if proto is not None else None
OH = proto["OH"].astype(np.float32) if proto is not None else None

with open(CACHE_DIR / "thresholds_proto_v1.json") as f:
    THRESH = json.load(f)

# same scorer as in Cell 3R
DOC = {"journal_banner","doc_fullpage","pure_text","book_cover"}
ICON= {"logo","qr","tiny","empty"}

def make_scorer(text_emb_all, labels, starts, ends, P=None, OH=None, mix=(0.55,0.85,0.75), beta=80.0):
    ranges = {lbl: (int(starts[i]), int(ends[i])) for i,lbl in enumerate(labels)}
    import torch
    def score(X: np.ndarray) -> np.ndarray:
        X = X.astype(np.float32)
        img = torch.from_numpy(X)
        sims = []
        for lbl in labels:
            a,b = ranges[lbl]
            T = torch.from_numpy(text_emb_all[a:b])
            sims.append((img @ T.T).max(dim=1).values.unsqueeze(1))
        text_logits = torch.cat(sims, dim=1)
        if P is None:
            out = text_logits
        else:
            P_t = torch.from_numpy(P); OH_t = torch.from_numpy(OH)
            proto_logits = torch.softmax(80.0 * (img @ P_t.T), dim=1) @ OH_t
            a_doc, a_icon, a_oth = mix
            A = []
            for lbl in labels:
                if lbl in DOC:   A.append(a_doc)
                elif lbl in ICON:A.append(a_icon)
                else:            A.append(a_oth)
            A = torch.tensor(A, dtype=torch.float32).view(1,-1)
            out = A * text_logits + (1.0 - A) * proto_logits
        # light doc rerank bump
        doc_idx = [i for i,l in enumerate(labels) if l in DOC]
        if doc_idx:
            sub = out[:, doc_idx]
            win = sub.softmax(dim=1).argmax(dim=1)
            for r in range(out.size(0)):
                out[r, doc_idx[win[r].item()]] += 1e-3
        return out.numpy()
    return score

scorer = make_scorer(text_emb_all, labels, starts, ends, P=P, OH=OH, mix=(0.55,0.85,0.75), beta=80.0)

# labels to filter out
FILTER_LABELS = {
    "qr","logo","journal_banner","doc_fullpage","pure_text","book_cover",
    "empty","tiny","chart","ecg","gel_electrophoresis","map","profile",
    "illustration","phylogenetic_tree","house"
}

# score in chunks
def score_all(chunk=4096):
    out = np.zeros((N, len(labels)), dtype=np.float32)
    for i in range(0, N, chunk):
        X = np.array(E[i:i+chunk], dtype=np.float32)
        out[i:i+chunk] = scorer(X)
    return out

S = score_all(chunk=4096)

# build dataframe of scores (optional, large file)
pred_rows = []
for i in range(N):
    rec = {"row": i, "path": index_df.loc[i, "path"]}
    for c,lbl in enumerate(labels): rec[lbl] = float(S[i, c])
    pred_rows.append(rec)
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(CACHE_DIR / "predictions_proto_v1.csv", index=False)
print("[INFO] wrote predictions_proto_v1.csv")

# apply thresholds — (A) “any label >= thresh” OR (B) “top1 in FILTER_LABELS and >= thresh[top1]”
th = {k: float(v) for k,v in THRESH.items()}
top1_idx = S.argmax(axis=1)
top1_lbl = np.array([labels[i] for i in top1_idx])
top1_val = S.max(axis=1)

any_hit = np.zeros(N, dtype=bool)
for ci,lbl in enumerate(labels):
    thr = th.get(lbl, 0.18)
    any_hit |= (S[:,ci] >= thr) & np.isin(lbl, list(FILTER_LABELS))

top1_hit = (np.isin(top1_lbl, list(FILTER_LABELS)) & (top1_val >= np.array([th.get(l,0.18) for l in top1_lbl])))

drop_mask = any_hit | top1_hit
pred_df["filter"] = drop_mask
keep_df = pred_df[~drop_mask].copy()
drop_df = pred_df[drop_mask].copy()

keep_df.to_csv(CACHE_DIR / "filtered_keep_proto_v1.csv", index=False)
drop_df.to_csv(CACHE_DIR / "filtered_drop_proto_v1.csv", index=False)
print(f"[RESULT] keep={len(keep_df)}  drop={len(drop_df)}  total={len(pred_df)}")

# quick class histogram of what you plan to drop (by top1)
drop_top1 = pd.Series(top1_lbl[drop_mask]).value_counts().to_frame("count")
drop_top1.to_csv(CACHE_DIR / "drop_top1_hist.csv")
print(drop_top1.head(10))

[INFO] wrote predictions_proto_v1.csv
[RESULT] keep=24296  drop=23749  total=48045
                     count
empty                17932
tiny                  4323
pure_text              795
gel_electrophoresis    129
journal_banner         122
logo                   116
map                    108
illustration            69
profile                 62
chart                   52


In [ ]:
# === Cell 5 — Materialize filter to filesystem (KEEP/DROP/UNSCORED) ===
import os, shutil
from pathlib import Path
from tqdm import tqdm
import pandas as pd

ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
IMAGES_DIR = ROOT / "kaggle/working_v2/rag_knowledge_base/images"
CACHE_DIR  = ROOT / "kaggle/working_v2/emb_cache_siglip"
OUT_ROOT   = ROOT / "kaggle/working_v2/filtered_siglip_v1"

# ---------- choose how to materialize ----------
MODE   = "symlink"   # "symlink" (safe & fast) | "copy" (dup files) | "move" (removes from source)
DRY_RUN = False      # True = just prepare manifests and folders, do not link/copy/move

# ---------- load keep/drop lists ----------
keep_df = pd.read_csv(CACHE_DIR / "filtered_keep_proto_v1.csv")
drop_df = pd.read_csv(CACHE_DIR / "filtered_drop_proto_v1.csv")
keep_set = set(keep_df["path"].astype(str))
drop_set = set(drop_df["path"].astype(str))

# ---------- enumerate all images to find unscored ----------
exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff",".gif"}
all_paths = [str(p) for p in IMAGES_DIR.rglob("*") if p.suffix.lower() in exts]
all_set = set(all_paths)
unscored = sorted(all_set - keep_set - drop_set)

# ---------- write manifests ----------
(OUT_ROOT / "manifests").mkdir(parents=True, exist_ok=True)
pd.Series(sorted(keep_set)).to_csv(OUT_ROOT / "manifests/keep.txt", index=False, header=False)
pd.Series(sorted(drop_set)).to_csv(OUT_ROOT / "manifests/drop.txt", index=False, header=False)
pd.Series(unscored).to_csv(OUT_ROOT / "manifests/unscored.txt", index=False, header=False)

KEEP_DIR = OUT_ROOT / "keep"
DROP_DIR = OUT_ROOT / "drop"
UNSCORED_DIR = OUT_ROOT / "unscored"
for d in [KEEP_DIR, DROP_DIR, UNSCORED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def relpath_within_root(p: str) -> Path:
    p = Path(p)
    try:
        return p.relative_to(IMAGES_DIR)
    except ValueError:
        # just in case a path isn't under IMAGES_DIR
        return Path("__external__") / p.name

def place(src: str, dst_root: Path):
    rel = relpath_within_root(src)
    dst = dst_root / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if DRY_RUN:
        return
    if MODE == "symlink":
        try:
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            os.symlink(src, dst)
        except FileExistsError:
            pass
    elif MODE == "copy":
        if not dst.exists():
            shutil.copy2(src, dst)
    elif MODE == "move":
        if not dst.exists():
            shutil.move(src, dst)
    else:
        raise ValueError("MODE must be one of: symlink, copy, move")

# ---------- apply ----------
for p in tqdm(sorted(keep_set), desc=f"KEEP -> {MODE}"):
    place(p, KEEP_DIR)
for p in tqdm(sorted(drop_set), desc=f"DROP -> {MODE}"):
    place(p, DROP_DIR)
for p in tqdm(unscored, desc=f"UNSCORED -> {MODE}"):
    place(p, UNSCORED_DIR)

print("Summary:")
print(" keep:", len(keep_set), " drop:", len(drop_set), " unscored:", len(unscored), " total:", len(all_set))
print("Output at:", OUT_ROOT)

# ---------- optional: generate a deletion script for DROP originals (review before running) ----------
sh = OUT_ROOT / "delete_drop.sh"
with open(sh, "w") as f:
    f.write("#!/usr/bin/env bash\nset -euo pipefail\n")
    f.write('while IFS= read -r p; do [ -f "$p" ] && rm -f "$p"; done < "%s"\n' % (OUT_ROOT/"manifests/drop.txt"))
os.chmod(sh, 0o755)
print("To DELETE dropped originals later (irreversible):")
print(f"  bash {sh}")


(Optional) Cell 5 — Physically separate files

In [ ]:
# Cell 5 — Move/symlink filtered files to KEEP/DROP folders (optional)

import shutil
from pathlib import Path
import pandas as pd

ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc")
CACHE_DIR  = ROOT / "kaggle/working_v2/emb_cache_siglip"
drop_df = pd.read_csv(CACHE_DIR / "filtered_drop_proto.csv")
keep_df = pd.read_csv(CACHE_DIR / "filtered_keep_proto.csv")

OUT_DIR = ROOT / "kaggle/working_v2/filtered_split"
KEEP_DIR = OUT_DIR / "keep"
DROP_DIR = OUT_DIR / "drop"
KEEP_DIR.mkdir(parents=True, exist_ok=True)
DROP_DIR.mkdir(parents=True, exist_ok=True)

USE_SYMLINKS = True   # set False to copy files

def place(row, dest_dir):
    src = Path(row["path"])
    dst = dest_dir / src.name
    try:
        if USE_SYMLINKS:
            if dst.exists(): dst.unlink()
            dst.symlink_to(src)
        else:
            if not dst.exists():
                shutil.copy2(src, dst)
    except Exception as e:
        print("skip", src, "->", e)

for _, r in drop_df.iterrows(): place(r, DROP_DIR)
for _, r in keep_df.iterrows(): place(r, KEEP_DIR)

print(f"[FILES] symlinked/copied: keep={len(keep_df)} drop={len(drop_df)} to {OUT_DIR}")


In [8]:
!uv  pip install iterative-stratification

Resolved 6 packages in 785ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)------------------     0 B/8.32 KiB        
Prepared 1 package in 46ms                                                   
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 7msation==0.1.9                      
 + iterative-stratification==0.1.9


In [11]:
import numpy as np, json
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, average_precision_score, precision_recall_curve
from sklearn.preprocessing import StandardScaler
import joblib

# Load cached features/labels
X = np.load("siglip_img_embeddings.npy")   # [N, D]
Y = np.load("siglip_labels.npy")           # [N, C]
labels = json.load(open("siglip_labels_schema.json"))

# Per-class threshold search to maximize F1 on val
def best_threshold(y_true, p):
    best_t, best_f1 = 0.5, 0.0
    prec, rec, thr = precision_recall_curve(y_true, p)
    grid = np.unique(np.concatenate([thr, np.linspace(0.05, 0.95, 19)]))
    for t in grid:
        yhat = (p >= t).astype(int)
        tp = (yhat & (y_true==1)).sum()
        fp = (yhat & (y_true==0)).sum()
        fn = ((1-yhat) & (y_true==1)).sum()
        if tp+fp == 0 or tp+fn == 0: 
            continue
        f1 = 2*tp / (2*tp + fp + fn)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

# Proper 5-fold CV
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []
all_thresholds = []

for fold, (tr_idx, va_idx) in enumerate(mskf.split(X, Y)):
    print(f"\n--- Fold {fold+1}/5 ---")
    Xtr, Xva = X[tr_idx], X[va_idx]
    Ytr, Yva = Y[tr_idx], Y[va_idx]
    
    # Drop constant labels in TRAIN (no variation) - fixes scikit-learn warning
    var_mask = (Ytr.sum(axis=0) > 0) & (Ytr.sum(axis=0) < len(Ytr))
    keep_idx = np.where(var_mask)[0]
    Xtr_filtered = Xtr
    Ytr_filtered = Ytr[:, keep_idx]
    Xva_filtered = Xva
    Yva_filtered = Yva[:, keep_idx]
    labels_filtered = [labels[i] for i in keep_idx]
    print(f"Kept {len(keep_idx)}/{len(var_mask)} labels after dropping constants")
    
    if len(keep_idx) == 0:
        print("No valid labels in this fold, skipping...")
        continue
    
    # Optional scaling (doesn't densify)
    scaler = StandardScaler(with_mean=False)
    Xtr_scaled = scaler.fit_transform(Xtr_filtered)
    Xva_scaled = scaler.transform(Xva_filtered)
    
    # Train OvR logistic regression (stable on small data)
    clf = OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",      # better than saga for tiny, imbalanced sets
            penalty="l2",
            C=2.0,
            max_iter=10000,
            class_weight="balanced",
        )
    )
    clf.fit(Xtr_scaled, Ytr_filtered)
    
    # Predict probs on val
    probs = clf.predict_proba(Xva_scaled)  # [Nva, C_filtered]
    
    # Per-class threshold search
    thresh = np.full(len(labels), 0.5)  # Default threshold for all original labels
    val_preds = np.zeros_like(Yva)
    per_class_ap = np.full(len(labels), np.nan)
    skipped = []
    
    for i, c in enumerate(keep_idx):
        lab = labels[c]
        y_true = Yva_filtered[:, i]
        if y_true.sum() == 0 or (len(y_true) - y_true.sum()) == 0:
            skipped.append(lab)
            continue
        p = probs[:, i]
        t, _ = best_threshold(y_true, p)
        thresh[c] = t
        val_preds[:, c] = (p >= t).astype(int)
        per_class_ap[c] = average_precision_score(y_true, p)
    
    # Metrics (only on labels that were trained)
    micro_f1 = f1_score(Yva_filtered, val_preds[:, keep_idx], average="micro", zero_division=0)
    macro_f1 = f1_score(Yva_filtered, val_preds[:, keep_idx], average="macro", zero_division=0)
    per_class_f1 = f1_score(Yva, val_preds, average=None, zero_division=0)
    mAP = np.nanmean(per_class_ap)
    
    print(f"Micro F1: {micro_f1:.3f}")
    print(f"Macro F1: {macro_f1:.3f}")
    print(f"mAP:      {mAP:.3f}")
    
    for lab, f1c, ap, t in zip(labels, per_class_f1, per_class_ap, thresh):
        if not np.isnan(ap):
            ap_str = f"{ap:.3f}"
        else:
            ap_str = "nan"
        print(f"{lab:22s}  F1={f1c:.3f}  AP={ap_str:>6}  thr={t:.2f}")
    
    if skipped:
        print("⚠️ Skipped (no pos/neg in val):", skipped)
    
    # Store fold results
    fold_results.append({
        'micro_f1': micro_f1,
        'macro_f1': macro_f1, 
        'mAP': mAP,
        'per_class_f1': per_class_f1,
        'per_class_ap': per_class_ap,
        'thresholds': thresh.copy(),
        'keep_idx': keep_idx.copy()
    })
    all_thresholds.append(thresh.copy())

# Average across folds
if fold_results:
    avg_micro = np.mean([r['micro_f1'] for r in fold_results])
    avg_macro = np.mean([r['macro_f1'] for r in fold_results])
    avg_mAP = np.mean([r['mAP'] for r in fold_results])
    
    print(f"\n=== 5-Fold CV Results ===")
    print(f"Avg Micro F1: {avg_micro:.3f} ± {np.std([r['micro_f1'] for r in fold_results]):.3f}")
    print(f"Avg Macro F1: {avg_macro:.3f} ± {np.std([r['macro_f1'] for r in fold_results]):.3f}")
    print(f"Avg mAP:      {avg_mAP:.3f} ± {np.std([r['mAP'] for r in fold_results]):.3f}")
    
    # Build robust per-label threshold averages (only where label was trained + had pos/neg)
    L = len(labels)
    thr_sums = np.zeros(L, dtype=float)
    thr_counts = np.zeros(L, dtype=int)
    
    for r in fold_results:
        thr = r["thresholds"]      # length = L, but only meaningful on r['keep_idx']
        keep = set(r["keep_idx"].tolist())
        valid_mask = ~np.isnan(r["per_class_ap"])  # True => had pos/neg
        for i in range(L):
            if (i in keep) and valid_mask[i]:
                thr_sums[i] += thr[i]
                thr_counts[i] += 1
    
    avg_thresholds = np.full(L, 0.5, dtype=float)
    mask = thr_counts > 0
    avg_thresholds[mask] = thr_sums[mask] / thr_counts[mask]
    
    # Average per-class metrics
    print(f"\nPer-class averages:")
    avg_per_class_f1 = np.nanmean([r['per_class_f1'] for r in fold_results], axis=0)
    avg_per_class_ap = np.nanmean([r['per_class_ap'] for r in fold_results], axis=0)
    
    for i, lab in enumerate(labels):
        f1_val = avg_per_class_f1[i] if not np.isnan(avg_per_class_f1[i]) else 0.0
        ap_val = avg_per_class_ap[i]
        ap_str = f"{ap_val:.3f}" if not np.isnan(ap_val) else "nan"
        thr_info = f"({thr_counts[i]} folds)" if thr_counts[i] > 0 else "(no valid folds)"
        print(f"{lab:22s}  F1={f1_val:.3f}  AP={ap_str:>6}  thr={avg_thresholds[i]:.2f} {thr_info}")

# Train final model on all data for deployment
print(f"\n=== Training Final Model on All Data ===")
# Drop constant labels from full dataset
var_mask_full = (Y.sum(axis=0) > 0) & (Y.sum(axis=0) < len(Y))
keep_idx_full = np.where(var_mask_full)[0]
X_final = X
Y_final = Y[:, keep_idx_full]
labels_final = [labels[i] for i in keep_idx_full]

# Scale features
scaler_final = StandardScaler(with_mean=False)
X_scaled_final = scaler_final.fit_transform(X_final)

# Train final classifier
clf_final = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=2.0,
        max_iter=10000,
        class_weight="balanced",
    )
)
clf_final.fit(X_scaled_final, Y_final)

# Use average thresholds from CV - IMPORTANT: only for kept labels, in same order
final_thresholds = avg_thresholds[keep_idx_full] if fold_results else np.full(len(keep_idx_full), 0.5)

# Persist model + scaler + thresholds for reuse
joblib.dump(clf_final, "siglip_linear_probe.joblib")
joblib.dump(scaler_final, "siglip_scaler.joblib")
np.save("siglip_thresholds.npy", final_thresholds)
np.save("siglip_keep_idx.npy", keep_idx_full)  # Save which labels were kept
json.dump(labels_final, open("siglip_labels_final.json", "w"))

print("Saved: siglip_linear_probe.joblib, siglip_scaler.joblib, siglip_thresholds.npy")
print(f"Final model trained on {len(keep_idx_full)}/{len(labels)} labels")


--- Fold 1/5 ---
Kept 15/16 labels after dropping constants
Micro F1: 0.829
Macro F1: 0.658
mAP:      0.897
qr                      F1=1.000  AP= 1.000  thr=0.95
logo                    F1=0.933  AP= 0.962  thr=0.05
chart                   F1=0.000  AP=   nan  thr=0.50
journal_banner          F1=1.000  AP= 1.000  thr=0.99
pure_text               F1=1.000  AP= 1.000  thr=0.55
book_cover              F1=0.000  AP=   nan  thr=0.50
empty                   F1=1.000  AP= 1.000  thr=0.95
ecg                     F1=0.000  AP=   nan  thr=0.50
profile                 F1=0.143  AP= 0.077  thr=0.00
gel_electrophoresis     F1=0.800  AP= 0.833  thr=0.10
map                     F1=1.000  AP= 1.000  thr=0.01
tiny                    F1=1.000  AP= 1.000  thr=0.04
house                   F1=1.000  AP= 1.000  thr=0.05
illustration            F1=0.000  AP=   nan  thr=0.50
phylogenetic_tree       F1=0.000  AP=   nan  thr=0.50
doc_fullpage            F1=1.000  AP= 1.000  thr=0.05
⚠️ Skipped (no pos/neg in v

/tmp/ipykernel_3174186/3960214466.py:157: RuntimeWarning: Mean of empty slice
  avg_per_class_ap = np.nanmean([r['per_class_ap'] for r in fold_results], axis=0)


Saved: siglip_linear_probe.joblib, siglip_scaler.joblib, siglip_thresholds.npy
Final model trained on 16/16 labels


In [12]:
import os, json, math, csv
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
from PIL import Image, ImageStat, UnidentifiedImageError
from tqdm import tqdm

import joblib
import torch
from transformers import AutoProcessor, AutoModel

# ===================== CONFIG =====================
# Đường dẫn thư mục gốc ảnh cần inference (duyệt đệ quy)
ROOT_DIR = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images"

# Đầu ra CSV
OUT_CSV  = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"

# Model SigLIP: dùng hub id hoặc local snapshot directory
# - Nếu bạn muốn bắt buộc dùng local snapshot:
#   MODEL = "/media/pc1/Ubuntu/Extend_Data/ngoc/models--google--siglip-so400m-patch14-384/snapshots/<commit_hash>"
MODEL = "google/siglip-so400m-patch14-384"
LOCAL_ONLY = False  # đặt True nếu môi trường offline, cache đã có sẵn

# Artifacts đã train
PROBE_PATH     = "siglip_linear_probe.joblib"
SCALER_PATH    = "siglip_scaler.joblib"
THRESH_PATH    = "siglip_thresholds.npy"
LABELS_FINAL   = "siglip_labels_final.json"

# Ảnh hợp lệ
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}

# Rule prefilters
MIN_TINY_SIDE = 24          # min(width,height) < 24 -> 'tiny'
EMPTY_WHITE_RATIO = 0.985   # nếu tỷ lệ điểm ảnh rất trắng vượt ngưỡng -> 'empty'
EMPTY_STD_LUMA   = 4.0      # hoặc độ lệch chuẩn độ sáng quá thấp -> 'empty'

# Hành động gợi ý theo nhãn
QUARANTINE_LABELS = {"tiny", "empty"}   # gặp các nhãn này -> quarantine
INDEX_LABELS      = None                # hoặc để None -> mặc định index nếu có >=1 nhãn
# ==================================================


# ---------- Utilities ----------
def iter_images(root: str):
    root = Path(root)
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            yield str(p)

def is_tiny(im: Image.Image) -> bool:
    w, h = im.size
    return min(w, h) < MIN_TINY_SIDE

def is_empty(im: Image.Image) -> bool:
    """
    Heuristic: ảnh gần như trắng hoặc độ biến thiên độ sáng rất thấp.
    """
    # Luma
    g = im.convert("L")
    arr = np.asarray(g, dtype=np.uint8)
    # tỉ lệ pixel rất trắng
    white_ratio = float((arr >= 250).mean())
    if white_ratio >= EMPTY_WHITE_RATIO:
        return True
    # độ lệch chuẩn của độ sáng
    std = float(arr.std())
    return std <= EMPTY_STD_LUMA

# ---------- Load artifacts ----------
labels: List[str] = json.load(open(LABELS_FINAL))
clf = joblib.load(PROBE_PATH)
scaler = joblib.load(SCALER_PATH)
thr = np.load(THRESH_PATH)                         # thứ tự khớp với labels_final
assert len(thr) == len(labels), "Thresholds and labels_final length mismatch!"

# Load SigLIP
processor = AutoProcessor.from_pretrained(MODEL, local_files_only=LOCAL_ONLY)
siglip = AutoModel.from_pretrained(MODEL, local_files_only=LOCAL_ONLY)
siglip.eval()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
siglip.to(DEVICE)

@torch.no_grad()
def embed_image(path: str) -> Optional[np.ndarray]:
    try:
        im = Image.open(path).convert("RGB")
    except (FileNotFoundError, UnidentifiedImageError, OSError):
        return None

    # Rule: if tiny -> có thể bỏ qua embedding để tiết kiệm compute
    if is_tiny(im):
        return np.zeros((siglip.config.vision_config.hidden_size,), dtype=np.float32)

    inputs = processor(images=im, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    feats = siglip.get_image_features(**inputs)                 # [1, D]
    feats = torch.nn.functional.normalize(feats, dim=-1)[0]     # [D]
    return feats.detach().cpu().numpy()

def predict_from_feature(fvec: np.ndarray) -> Tuple[Dict[str, float], List[str]]:
    # Chuẩn hoá theo scaler đã fit
    f_scaled = scaler.transform(fvec[None, :])       # (1, D)
    probs = clf.predict_proba(f_scaled)[0]           # (C,)
    picks = [lab for lab, p in zip(labels, probs) if p >= thr[labels.index(lab)]]
    return {lab: float(p) for lab, p in zip(labels, probs)}, picks

def decide_action(picks: List[str], rules: Dict[str, bool]) -> str:
    # rules: {"tiny": bool, "empty": bool}
    if rules.get("tiny") or rules.get("empty"):
        return "quarantine"
    if INDEX_LABELS is None:
        return "index" if len(picks) > 0 else "review"
    return "index" if any(lbl in picks for lbl in INDEX_LABELS) else "review"

# ---------- Inference loop ----------
rows = []
for img_path in tqdm(list(iter_images(ROOT_DIR)), desc="Inferencing"):
    try:
        im = Image.open(img_path).convert("RGB")
    except Exception:
        rows.append({
            "path": img_path, "status": "unreadable"
        })
        continue

    # Rule prefilters
    tiny_flag  = is_tiny(im)
    empty_flag = (not tiny_flag) and is_empty(im)   # nếu tiny rồi thì khỏi check empty
    rule_picks = []
    rule_probs = {}

    if "tiny" in labels and tiny_flag:
        rule_picks.append("tiny")
        rule_probs["tiny"] = 1.0
    if "empty" in labels and empty_flag:
        rule_picks.append("empty")
        rule_probs["empty"] = 1.0

    # Nếu chỉ muốn rule quyết định và bỏ qua model cho tiny/empty:
    run_model = not (tiny_flag or empty_flag)

    if run_model:
        f = embed_image(img_path)
        if f is None:
            rows.append({"path": img_path, "status": "embed_failed"})
            continue
        probs, picks = predict_from_feature(f)
    else:
        # Không chạy model: xác suất các label khác đặt 0
        probs = {lab: 0.0 for lab in labels}
        for k, v in rule_probs.items():
            probs[k] = v
        picks = list(set(rule_picks))  # từ rules

    action = decide_action(picks, {"tiny": tiny_flag, "empty": empty_flag})

    # Ghi dòng kết quả (flatten probs)
    row = {
        "path": img_path,
        "status": "ok",
        "action": action,
        "picked_labels": ",".join(picks)
    }
    for lab in labels:
        row[f"prob_{lab}"] = probs.get(lab, 0.0)
    rows.append(row)

# ---------- Save CSV ----------
fieldnames = ["path", "status", "action", "picked_labels"] + [f"prob_{lab}" for lab in labels]
with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for r in rows:
        w.writerow(r)

print(f"Saved inference CSV to: {OUT_CSV}")
print(f"Total images processed: {len(rows)}")


Inferencing: 100%|██████████| 2302/2302 [02:48<00:00, 13.66it/s] 

Saved inference CSV to: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv
Total images processed: 2302


filter

In [ ]:
import os
import csv
from pathlib import Path
from tqdm import tqdm
import shutil

# ==== CONFIG ====
CSV_PATH = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"

# Thư mục gốc hiện có chứa toàn bộ images (để tính đường dẫn tương đối)
IMAGES_ROOT = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")

# Thư mục đích (mới) để chứa ảnh bị lọc
DEST_ROOT   = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered")

# Chọn action nào sẽ “lọc” (ví dụ: chỉ chuyển quarantine)
ACTIONS_TO_FILTER = {"quarantine"}       # có thể thêm {"review"} nếu muốn

# Move hay Copy (True = move, False = copy)
MOVE_FILES = True

# Optional: giữ đúng tree thư mục con
PRESERVE_TREE = True

# Optional: thử hardlink để nhanh & tiết kiệm dung lượng (cùng filesystem).
# Nếu hardlink fail thì fallback sang copy.
USE_HARDLINK_IF_POSSIBLE = False
# ==============


DEST_ROOT.mkdir(parents=True, exist_ok=True)

moved, skipped, missing = 0, 0, 0

with open(CSV_PATH, "r", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for r in tqdm(rows, desc="Filtering images"):
    path = r.get("path", "")
    action = r.get("action", "")
    status = r.get("status", "")

    if action not in ACTIONS_TO_FILTER:
        skipped += 1
        continue
    if status != "ok":
        missing += 1
        continue

    src = Path(path)
    if not src.exists():
        missing += 1
        continue

    # Xác định đích:
    # - Nếu PRESERVE_TREE: giữ cấu trúc tương đối so với IMAGES_ROOT
    # - Ngược lại: dồn hết vào một folder phẳng
    if PRESERVE_TREE:
        try:
            rel = src.relative_to(IMAGES_ROOT)
        except ValueError:
            # Nếu ảnh không nằm dưới IMAGES_ROOT, chỉ lấy tên file
            rel = Path(src.name)
        dst = DEST_ROOT / action / rel
    else:
        dst = DEST_ROOT / action / src.name

    dst.parent.mkdir(parents=True, exist_ok=True)

    # Di chuyển hoặc copy
    try:
        if MOVE_FILES:
            # move sẽ tự tạo overwrite? -> mặc định không. Xử lý nếu file đã tồn tại
            if dst.exists():
                # Nếu file đã tồn tại đích, đổi tên tránh ghi đè
                dst = dst.with_name(dst.stem + "__dup" + dst.suffix)
            shutil.move(str(src), str(dst))
        else:
            if USE_HARDLINK_IF_POSSIBLE:
                try:
                    os.link(src, dst)   # hardlink
                except Exception:
                    shutil.copy2(src, dst)  # fallback copy
            else:
                shutil.copy2(src, dst)
        moved += 1
    except Exception as e:
        print(f"⚠️ Lỗi khi chuyển: {src} -> {dst}: {e}")

print(f"\nDone. moved/copied: {moved}, skipped (action not in {ACTIONS_TO_FILTER}): {skipped}, missing/bad: {missing}")
print(f"Output root: {DEST_ROOT}")


Filtering images: 100%|██████████| 2302/2302 [00:00<00:00, 10371.26it/s]


Done. moved/copied: 473, skipped (action not in {'review', 'quarantine'}): 1028, missing/bad: 801
Output root: /media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered


In [17]:
import csv
from pathlib import Path
import shutil

# ==== CONFIG ====
CSV_PATH     = "/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/results/inference_results.csv"
IMAGES_ROOT  = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")
FILTERED_ROOT= Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered")
ACTION       = "review"     # khôi phục ảnh bị gán action này
DRY_RUN      = False         # True: chỉ in; False: move thật
# ===============

def rel_from_images_root(p: Path) -> Path:
    try:
        return p.relative_to(IMAGES_ROOT)
    except ValueError:
        return Path(p.name)  # fallback khi không cùng gốc

rows = []
with open(CSV_PATH, "r", newline="") as f:
    rows = list(csv.DictReader(f))

moved = skipped = missing = 0
for r in rows:
    if r.get("action") != ACTION or r.get("status") != "ok":
        skipped += 1
        continue

    orig_path = Path(r["path"])
    rel = rel_from_images_root(orig_path)

    # đường dẫn hiện tại của file trong filtered/
    src = FILTERED_ROOT / ACTION / rel
    # đích khôi phục là đúng đường dẫn gốc
    dst = orig_path

    if not src.exists():
        # fallback: có thể lúc move trước đó không preserve tree, thử theo tên file
        alt = FILTERED_ROOT / ACTION / src.name
        if alt.exists():
            src = alt
        else:
            print(f"⚠️ Không tìm thấy file trong filtered: {src}")
            missing += 1
            continue

    if not DRY_RUN:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            # tránh ghi đè (giữ bản khôi phục)
            dst = dst.with_name(dst.stem + "__restored" + dst.suffix)
        shutil.move(str(src), str(dst))
    else:
        print(f"[DRY] move {src} -> {dst}")

    moved += 1

print(f"\nDone. moved={moved}, skipped={skipped}, missing={missing}")
print(f"DRY_RUN={DRY_RUN} | FILTERED_ROOT={FILTERED_ROOT} | IMAGES_ROOT={IMAGES_ROOT}")



Done. moved=473, skipped=1829, missing=0
DRY_RUN=False | FILTERED_ROOT=/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/filtered | IMAGES_ROOT=/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images


In [ ]:
from pathlib import Path

root = Path("/media/pc1/Ubuntu/Extend_Data/ngoc/kaggle/working/rag_knowledge_base/images")
count = sum(1 for _ in root.rglob("*") if _.is_file())
print(f"Tổng số file: {count}")


Tổng số file: 1501


: 